In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

# ==========================================
# 1. REPRESENTATIVE MATERIAL PROPERTIES (SI)
# ==========================================
COLUMN_MATERIALS = {
    "Paper (Cardstock / A4 Roll)": {"E": 2.2e9, "sigma_c": 25e6, "rho": 750.0},
    "Balsa Wood (Light/Medium)":   {"E": 3.5e9, "sigma_c": 12e6, "rho": 160.0},
    "3D-Printed PLA (Standard)":     {"E": 2.8e9, "sigma_c": 45e6, "rho": 1240.0},
}

BOUNDARY_CONDITIONS = {
    "Pinned-Pinned (K = 1.0)":  1.0,
    "Fixed-Pinned (K = 0.7)":   0.7,
    "Fixed-Fixed (K = 0.5)":    0.5,
    "Cantilever / Free (K = 2.0)": 2.0,
}

# ==========================================
# 2. SIMPLE EULER BUCKLING MODEL & SOLVER
# ==========================================
def solve_column_buckling(mat_name, section_type, L_mm, d1_mm, d2_mm, wall_mm, K_val, P_exp_N):
    mat = COLUMN_MATERIALS[mat_name]
    E = mat["E"]
    sigma_c = mat["sigma_c"]

    L = L_mm * 1e-3
    Le = K_val * L

    if section_type == "Hollow Tube":
        # Outer diameter d1_mm, wall thickness wall_mm
        d_out = d1_mm * 1e-3
        t = min(wall_mm * 1e-3, 0.45 * d_out)
        d_in = max(d_out - 2.0 * t, 0.0)
        A = (np.pi / 4.0) * (d_out**2 - d_in**2)
        I = (np.pi / 64.0) * (d_out**4 - d_in**4)
        geom_desc = f"Outer Ø={d1_mm:.1f} mm, Wall t={t*1e3:.2f} mm"

    elif section_type == "Rectangular Strip":
        # Width d1_mm (b), thickness d2_mm (h) -> buckling about weak axis (h)
        b = d1_mm * 1e-3
        h = d2_mm * 1e-3
        A = b * h
        I = (b * h**3) / 12.0
        geom_desc = f"Width b={d1_mm:.1f} mm, Thickness h={d2_mm:.1f} mm"

    else:  # "Solid Square"
        a = d1_mm * 1e-3
        A = a**2
        I = (a**4) / 12.0
        geom_desc = f"Square side a={d1_mm:.1f} mm"

    k_gyr = np.sqrt(I / max(A, 1e-15))
    slenderness_eff = Le / max(k_gyr, 1e-9)

    # Simple model: Euler critical load vs. direct crushing limit
    P_euler = (np.pi**2 * E * I) / max(Le**2, 1e-15)
    P_crush = sigma_c * A
    P_model = min(P_euler, P_crush)
    governing_model = "Euler Buckling" if P_euler <= P_crush else "Direct Compression Yield"

    diff_N = P_exp_N - P_model
    diff_pct = (diff_N / max(P_model, 1e-12)) * 100.0
    knockdown = P_exp_N / max(P_model, 1e-12)

    return {
        "L_mm": L_mm,
        "Le_mm": Le * 1e3,
        "A_mm2": A * 1e6,
        "I_mm4": I * 1e12,
        "k_mm": k_gyr * 1e3,
        "Slenderness": slenderness_eff,
        "P_euler_N": P_euler,
        "P_crush_N": P_crush,
        "P_model_N": P_model,
        "P_exp_N": P_exp_N,
        "Diff_N": diff_N,
        "Diff_pct": diff_pct,
        "Knockdown": knockdown,
        "Governing": governing_model,
        "Geom_Desc": geom_desc,
    }

# ==========================================
# 3. PLOTS: MODEL VS EXP & 2D BUCKLED SHAPE
# ==========================================
def plot_buckling_comparison(res, mat_name, section_type, bc_label):
    plt.style.use("seaborn-v0_8-whitegrid")
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13.2, 4.5))

    # Left: Simple Model vs Experimental Load
    labels = ["Simple Model\n(P_model)", "Experimental\n(P_exp)"]
    vals = [res["P_model_N"], res["P_exp_N"]]
    cols = ["#2563eb", "#f97316"]

    bars = ax1.bar(labels, vals, color=cols, edgecolor="#0f172a", linewidth=1.2, width=0.48, alpha=0.92)
    ax1.set_title(f"Buckling Load Comparison — {mat_name}", fontsize=11, fontweight="bold", pad=10)
    ax1.set_ylabel("Critical Load [N]", fontsize=9.5)
    max_v = max(vals) if max(vals) > 0 else 1.0
    ax1.set_ylim(0, max_v * 1.18)

    for bar, v in zip(bars, vals):
        ax1.text(
            bar.get_x() + bar.get_width() / 2.0,
            v + max_v * 0.025,
            f"{v:.2f} N",
            ha="center", va="bottom", fontsize=9.5, fontweight="bold", color="#0f172a"
        )

    ax1.annotate(
        f"Difference: {res['Diff_pct']:+.1f}%\n(Exp/Model = {res['Knockdown']:.2f})",
        xy=(1, res["P_exp_N"]),
        xytext=(0.55, max_v * 0.82),
        fontsize=8.5,
        bbox=dict(boxstyle="round,pad=0.35", facecolor="#f8fafc", edgecolor="#94a3b8"),
        arrowprops=dict(arrowstyle="->", color="#475569", lw=1.0)
    )

    # Right: 2D Buckled Column Mode Shape (normalized lateral deflection)
    s = np.linspace(0.0, 1.0, 200)
    z_mm = s * res["L_mm"]

    if "Pinned-Pinned" in bc_label:
        w_norm = np.sin(np.pi * s)
    elif "Fixed-Fixed" in bc_label:
        w_norm = 0.5 * (1.0 - np.cos(2.0 * np.pi * s))
    elif "Cantilever" in bc_label:
        w_norm = 1.0 - np.cos(0.5 * np.pi * s)
    else:  # Fixed-Pinned approximate first mode
        w_norm = np.sin(np.pi * s) - 0.25 * np.sin(2.0 * np.pi * s)
        w_norm = w_norm / max(np.max(np.abs(w_norm)), 1e-9)

    amp_mm = max(0.08 * res["L_mm"], 4.0)
    ax2.plot([0, 0], [0, res["L_mm"]], color="#94a3b8", linestyle="--", linewidth=1.2, label="Undeformed Axis")
    ax2.plot(w_norm * amp_mm, z_mm, color="#dc2626", linewidth=2.5, label="Buckled Shape (Illustrative)")
    ax2.scatter([0, 0], [0, res["L_mm"]], color="#0f172a", s=40, zorder=5)

    ax2.set_title(f"2D Buckling Mode — {section_type} ({bc_label})", fontsize=10.5, fontweight="bold", pad=10)
    ax2.set_xlabel("Lateral Deflection w [mm] (Illustrative Scale)", fontsize=9)
    ax2.set_ylabel("Column Height z [mm]", fontsize=9.5)
    ax2.legend(fontsize=8.5, frameon=True, facecolor="white", edgecolor="#cbd5e1")
    ax2.set_xlim(-amp_mm * 1.35, amp_mm * 1.35)

    plt.tight_layout()
    plt.show()

# ==========================================
# 4. EXPLANATION CARD (WHY MODEL & EXP DIFFER)
# ==========================================
def render_comparison_summary(res, mat_name, section_type, bc_label):
    diff_pct = res["Diff_pct"]
    if abs(diff_pct) <= 15.0:
        badge = "<span style='background:#065f46;color:#a7f3d0;padding:3px 10px;border-radius:999px;font-size:11px;font-weight:700;'>CLOSE AGREEMENT WITH SIMPLE MODEL</span>"
        border_col = "#10b981"
        agreement_msg = (
            f"Experimental buckling load (<b>{res['P_exp_N']:.2f} N</b>) is within <b>±15%</b> "
            f"of the simple theoretical model (<b>{res['P_model_N']:.2f} N</b>, difference: <b>{diff_pct:+.1f}%</b>)."
        )
    elif diff_pct < -15.0:
        badge = "<span style='background:#991b1b;color:#fecaca;padding:3px 10px;border-radius:999px;font-size:11px;font-weight:700;'>EXPERIMENTAL LOWER THAN MODEL</span>"
        border_col = "#ef4444"
        agreement_msg = (
            f"Experimental load (<b>{res['P_exp_N']:.2f} N</b>) is <b>{abs(diff_pct):.1f}% lower</b> "
            f"than the simple model (<b>{res['P_model_N']:.2f} N</b>)."
        )
    else:
        badge = "<span style='background:#92400e;color:#fde68a;padding:3px 10px;border-radius:999px;font-size:11px;font-weight:700;'>EXPERIMENTAL HIGHER THAN MODEL</span>"
        border_col = "#f59e0b"
        agreement_msg = (
            f"Experimental load (<b>{res['P_exp_N']:.2f} N</b>) is <b>{diff_pct:+.1f}% higher</b> "
            f"than the simple model (<b>{res['P_model_N']:.2f} N</b>)."
        )

    # Tailored physical reasons depending on material & section
    reasons = [
        "<b>Initial imperfections &amp; load eccentricity:</b> Slight initial crookedness or off-center axial load induces bending moment early, typically reducing experimental buckling load below ideal Euler theory.",
        f"<b>Boundary condition realism ({bc_label}):</b> Ideal K factors assume frictionless pins or rigid clamps; real fixtures often have rotational compliance or slip.",
    ]
    if "Paper" in mat_name:
        reasons.append("<b>Paper fiber &amp; seam/roll effects:</b> Grain direction, humidity, overlapping seams, or creasing alter effective stiffness EI and local wrinkling.")
    elif "Balsa" in mat_name:
        reasons.append("<b>Balsa grain &amp; density scatter:</b> Balsa properties vary strongly with grain alignment and local density; low shear modulus can also slightly reduce effective buckling load in short columns.")
    elif "3D-Printed" in mat_name:
        reasons.append("<b>3D-print anisotropy &amp; infill:</b> Layer adhesion, print orientation, and infill pattern cause orthotropic stiffness rather than homogeneous isotropic behavior.")

    if section_type == "Hollow Tube":
        reasons.append("<b>Thin-wall local ovalization/buckling:</b> Thin walls can distort locally before global Euler buckling completes.")

    reasons_html = "".join(f"<li style='margin-bottom:4px;'>{r}</li>" for r in reasons)

    html = f"""
    <div style="
        background: linear-gradient(135deg, #0f172a 0%, #1e293b 100%);
        border: 1px solid #334155;
        border-left: 4px solid {border_col};
        border-radius: 8px;
        padding: 14px 18px;
        margin-top: 10px;
        color: #f8fafc;
        font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;
    ">
        <div style="display:flex;justify-content:space-between;align-items:center;margin-bottom:8px;">
            <div style="font-size:13px;font-weight:700;">Model vs. Experiment Comparison &amp; Why They Differ</div>
            {badge}
        </div>
        <div style="font-size:13px;line-height:1.55;color:#e2e8f0;margin-bottom:8px;">
            {agreement_msg} Governing simple mechanism: <b>{res['Governing']}</b>
            (Effective slenderness <b>K·L/k = {res['Slenderness']:.1f}</b>, I = <b>{res['I_mm4']:.1f} mm⁴</b>).
        </div>
        <div style="font-size:12.5px;color:#cbd5e1;">
            <b>Key reasons for differences between simple model and test:</b>
            <ul style="margin:6px 0 0 18px;padding:0;line-height:1.45;">{reasons_html}</ul>
        </div>
    </div>
    """
    display(HTML(html))

# ==========================================
# 5. INTERACTIVE ENGLISH UI
# ==========================================
style = {"description_width": "140px"}
layout_w = widgets.Layout(width="330px")

mat_dd = widgets.Dropdown(
    options=list(COLUMN_MATERIALS.keys()),
    value="Paper (Cardstock / A4 Roll)",
    description="Column Material:",
    style=style, layout=layout_w
)

sec_dd = widgets.Dropdown(
    options=["Hollow Tube", "Rectangular Strip", "Solid Square"],
    value="Hollow Tube",
    description="Cross-Section:",
    style=style, layout=layout_w
)

bc_dd = widgets.Dropdown(
    options=list(BOUNDARY_CONDITIONS.keys()),
    value="Pinned-Pinned (K = 1.0)",
    description="End Condition:",
    style=style, layout=layout_w
)

L_sl = widgets.FloatSlider(
    value=250.0, min=80.0, max=600.0, step=10.0,
    description="Length L [mm]:",
    readout_format=".0f", style=style, layout=layout_w
)

d1_sl = widgets.FloatSlider(
    value=20.0, min=4.0, max=80.0, step=1.0,
    description="Dim 1 (Ø / b / a) [mm]:",
    readout_format=".1f", style=style, layout=layout_w
)

d2_sl = widgets.FloatSlider(
    value=2.0, min=0.5, max=30.0, step=0.5,
    description="Dim 2 (Strip h) [mm]:",
    readout_format=".1f", style=style, layout=layout_w
)

wall_sl = widgets.FloatSlider(
    value=0.6, min=0.1, max=6.0, step=0.1,
    description="Tube Wall t [mm]:",
    readout_format=".1f", style=style, layout=layout_w
)

p_exp_sl = widgets.FloatSlider(
    value=18.0, min=0.5, max=300.0, step=0.5,
    description="Measured P_exp [N]:",
    readout_format=".1f", style=style, layout=layout_w
)

run_btn = widgets.Button(
    description="Compare Model vs. Test",
    button_style="primary",
    icon="bar-chart",
    layout=widgets.Layout(width="200px", height="36px")
)

out = widgets.Output()

def toggle_geom_inputs(*args):
    st = sec_dd.value
    wall_sl.layout.display = "flex" if st == "Hollow Tube" else "none"
    d2_sl.layout.display = "flex" if st == "Rectangular Strip" else "none"
    if st == "Hollow Tube":
        d1_sl.description = "Outer Dia. D [mm]:"
    elif st == "Rectangular Strip":
        d1_sl.description = "Strip Width b [mm]:"
    else:
        d1_sl.description = "Square Side a [mm]:"

sec_dd.observe(toggle_geom_inputs, names="value")
toggle_geom_inputs()

def run_buckling_ui(*args):
    with out:
        clear_output(wait=True)
        K_val = BOUNDARY_CONDITIONS[bc_dd.value]
        res = solve_column_buckling(
            mat_dd.value,
            sec_dd.value,
            L_sl.value,
            d1_sl.value,
            d2_sl.value,
            wall_sl.value,
            K_val,
            p_exp_sl.value
        )

        plot_buckling_comparison(res, mat_dd.value, sec_dd.value, bc_dd.value)

        summary_tbl = pd.DataFrame([{
            "Geometry": res["Geom_Desc"],
            "Area [mm²]": round(res["A_mm2"], 2),
            "I [mm⁴]": round(res["I_mm4"], 1),
            "K·L/k": round(res["Slenderness"], 1),
            "Euler P_cr [N]": round(res["P_euler_N"], 2),
            "Model P_model [N]": round(res["P_model_N"], 2),
            "Measured P_exp [N]": round(res["P_exp_N"], 2),
            "Diff [%]": round(res["Diff_pct"], 1),
        }])
        display(summary_tbl.style.hide(axis="index"))

        render_comparison_summary(res, mat_dd.value, sec_dd.value, bc_dd.value)

run_btn.on_click(run_buckling_ui)

ui = widgets.VBox([
    widgets.HTML("""
        <div style="padding:10px 14px;background:#0f172a;border-radius:8px;border:1px solid #334155;margin-bottom:8px;">
            <div style="font-size:17px;font-weight:700;color:#f8fafc;">Column Buckling Tester — Paper, Balsa &amp; 3D-Printed vs. Simple Model</div>
            <div style="font-size:12px;color:#94a3b8;">Enter column geometry, boundary condition &amp; experimental load, then compare with Euler model</div>
        </div>
    """),
    widgets.HBox([
        widgets.VBox([mat_dd, sec_dd, bc_dd, p_exp_sl], layout=widgets.Layout(margin="0 14px 0 0")),
        widgets.VBox([L_sl, d1_sl, d2_sl, wall_sl], layout=widgets.Layout(margin="0 14px 0 0")),
        widgets.VBox([run_btn], layout=widgets.Layout(justify_content="center", padding="8px"))
    ], layout=widgets.Layout(align_items="flex-start"))
])

display(ui, out)
run_buckling_ui()

Output()

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

# ==========================================
# 1. REALISTIC REPRESENTATIVE MATERIAL DATA
# ==========================================
COLUMN_MATERIALS = {
    "Paper (Cardstock / A4 Roll)": {
        "E": 2.2e9, "sigma_c": 22e6, "knockdown": 0.72,
        "note": "Typical rolled/cardstock effective E ~ 2.2 GPa; seams/creases & eccentricity typically yield ~0.65–0.80 of Euler load."
    },
    "Balsa Wood (Light/Medium)": {
        "E": 3.5e9, "sigma_c": 12e6, "knockdown": 0.78,
        "note": "Typical longitudinal E ~ 3.5 GPa; grain variation & slight load offset often give ~0.70–0.85 of Euler load."
    },
    "3D-Printed PLA (Standard)": {
        "E": 2.8e9, "sigma_c": 45e6, "knockdown": 0.82,
        "note": "Typical FDM PLA E ~ 2.8 GPa; layer/infill anisotropy & minor imperfections typically give ~0.75–0.90 of Euler load."
    },
}

BOUNDARY_CONDITIONS = {
    "Pinned-Pinned (K = 1.0)":   1.0,
    "Fixed-Pinned (K = 0.7)":    0.7,
    "Fixed-Fixed (K = 0.5)":     0.5,
    "Cantilever / Free (K = 2.0)": 2.0,
}

# ==========================================
# 2. SECTION & EULER BUCKLING MODEL
# ==========================================
def compute_section_props(section_type, d1_mm, d2_mm, wall_mm):
    if section_type == "Hollow Tube":
        d_out = d1_mm * 1e-3
        t = min(wall_mm * 1e-3, 0.45 * d_out)
        d_in = max(d_out - 2.0 * t, 0.0)
        A = (np.pi / 4.0) * (d_out**2 - d_in**2)
        I = (np.pi / 64.0) * (d_out**4 - d_in**4)
        geom_desc = f"Outer Ø={d1_mm:.1f} mm, Wall t={t*1e3:.2f} mm"
    elif section_type == "Rectangular Strip":
        b = d1_mm * 1e-3
        h = d2_mm * 1e-3
        A = b * h
        I = (b * h**3) / 12.0
        geom_desc = f"Width b={d1_mm:.1f} mm, Thickness h={d2_mm:.1f} mm"
    else:  # "Solid Square"
        a = d1_mm * 1e-3
        A = a**2
        I = (a**4) / 12.0
        geom_desc = f"Square side a={d1_mm:.1f} mm"
    return A, I, geom_desc

def realistic_test_load_N(mat_name, section_type, L_mm, d1_mm, d2_mm, wall_mm, K_val):
    mat = COLUMN_MATERIALS[mat_name]
    A, I, _ = compute_section_props(section_type, d1_mm, d2_mm, wall_mm)
    Le = K_val * (L_mm * 1e-3)
    P_euler = (np.pi**2 * mat["E"] * I) / max(Le**2, 1e-15)
    P_crush = mat["sigma_c"] * A
    P_ideal = min(P_euler, P_crush)
    # Realistic physical test load incorporating typical knockdown + deterministic slight variance
    geom_factor = 0.95 if section_type == "Hollow Tube" else (0.90 if section_type == "Rectangular Strip" else 0.96)
    p_exp = P_ideal * mat["knockdown"] * geom_factor
    return float(np.clip(round(p_exp, 1), 0.5, 500.0))

def solve_column_buckling(mat_name, section_type, L_mm, d1_mm, d2_mm, wall_mm, K_val, P_exp_N):
    mat = COLUMN_MATERIALS[mat_name]
    E = mat["E"]
    sigma_c = mat["sigma_c"]

    L = L_mm * 1e-3
    Le = K_val * L
    A, I, geom_desc = compute_section_props(section_type, d1_mm, d2_mm, wall_mm)

    k_gyr = np.sqrt(I / max(A, 1e-15))
    slenderness_eff = Le / max(k_gyr, 1e-9)

    P_euler = (np.pi**2 * E * I) / max(Le**2, 1e-15)
    P_crush = sigma_c * A
    P_model = min(P_euler, P_crush)
    governing_model = "Euler Buckling" if P_euler <= P_crush else "Direct Compression Yield"

    diff_N = P_exp_N - P_model
    diff_pct = (diff_N / max(P_model, 1e-12)) * 100.0
    knockdown = P_exp_N / max(P_model, 1e-12)

    return {
        "L_mm": L_mm,
        "Le_mm": Le * 1e3,
        "A_mm2": A * 1e6,
        "I_mm4": I * 1e12,
        "k_mm": k_gyr * 1e3,
        "Slenderness": slenderness_eff,
        "P_euler_N": P_euler,
        "P_crush_N": P_crush,
        "P_model_N": P_model,
        "P_exp_N": P_exp_N,
        "Diff_N": diff_N,
        "Diff_pct": diff_pct,
        "Knockdown": knockdown,
        "Governing": governing_model,
        "Geom_Desc": geom_desc,
    }

# ==========================================
# 3. PLOTS: MODEL VS EXP & 2D BUCKLED SHAPE
# ==========================================
def plot_buckling_comparison(res, mat_name, section_type, bc_label):
    plt.style.use("seaborn-v0_8-whitegrid")
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13.2, 4.5))

    labels = ["Simple Model\n(P_model)", "Experimental\n(P_exp)"]
    vals = [res["P_model_N"], res["P_exp_N"]]
    cols = ["#2563eb", "#f97316"]

    bars = ax1.bar(labels, vals, color=cols, edgecolor="#0f172a", linewidth=1.2, width=0.48, alpha=0.92)
    ax1.set_title(f"Buckling Load Comparison — {mat_name}", fontsize=11, fontweight="bold", pad=10)
    ax1.set_ylabel("Critical Load [N]", fontsize=9.5)
    max_v = max(vals) if max(vals) > 0 else 1.0
    ax1.set_ylim(0, max_v * 1.20)

    for bar, v in zip(bars, vals):
        ax1.text(
            bar.get_x() + bar.get_width() / 2.0,
            v + max_v * 0.025,
            f"{v:.2f} N",
            ha="center", va="bottom", fontsize=9.5, fontweight="bold", color="#0f172a"
        )

    ax1.annotate(
        f"Difference: {res['Diff_pct']:+.1f}%\n(Exp/Model = {res['Knockdown']:.2f})",
        xy=(1, res["P_exp_N"]),
        xytext=(0.48, max_v * 0.84),
        fontsize=8.5,
        bbox=dict(boxstyle="round,pad=0.35", facecolor="#f8fafc", edgecolor="#94a3b8"),
        arrowprops=dict(arrowstyle="->", color="#475569", lw=1.0)
    )

    s = np.linspace(0.0, 1.0, 200)
    z_mm = s * res["L_mm"]

    if "Pinned-Pinned" in bc_label:
        w_norm = np.sin(np.pi * s)
    elif "Fixed-Fixed" in bc_label:
        w_norm = 0.5 * (1.0 - np.cos(2.0 * np.pi * s))
    elif "Cantilever" in bc_label:
        w_norm = 1.0 - np.cos(0.5 * np.pi * s)
    else:
        w_norm = np.sin(np.pi * s) - 0.25 * np.sin(2.0 * np.pi * s)
        w_norm = w_norm / max(np.max(np.abs(w_norm)), 1e-9)

    amp_mm = max(0.08 * res["L_mm"], 4.0)
    ax2.plot([0, 0], [0, res["L_mm"]], color="#94a3b8", linestyle="--", linewidth=1.2, label="Undeformed Axis")
    ax2.plot(w_norm * amp_mm, z_mm, color="#dc2626", linewidth=2.5, label="Buckled Shape (Illustrative)")
    ax2.scatter([0, 0], [0, res["L_mm"]], color="#0f172a", s=40, zorder=5)

    ax2.set_title(f"2D Buckling Mode — {section_type} ({bc_label})", fontsize=10.5, fontweight="bold", pad=10)
    ax2.set_xlabel("Lateral Deflection w [mm] (Illustrative Scale)", fontsize=9)
    ax2.set_ylabel("Column Height z [mm]", fontsize=9.5)
    ax2.legend(fontsize=8.5, frameon=True, facecolor="white", edgecolor="#cbd5e1")
    ax2.set_xlim(-amp_mm * 1.35, amp_mm * 1.35)

    plt.tight_layout()
    plt.show()

# ==========================================
# 4. EXPLANATION CARD
# ==========================================
def render_comparison_summary(res, mat_name, section_type, bc_label):
    diff_pct = res["Diff_pct"]
    mat_note = COLUMN_MATERIALS[mat_name]["note"]

    if abs(diff_pct) <= 15.0:
        badge = "<span style='background:#065f46;color:#a7f3d0;padding:3px 10px;border-radius:999px;font-size:11px;font-weight:700;'>CLOSE AGREEMENT</span>"
        border_col = "#10b981"
    elif diff_pct < -15.0:
        badge = "<span style='background:#991b1b;color:#fecaca;padding:3px 10px;border-radius:999px;font-size:11px;font-weight:700;'>EXPERIMENTAL LOWER THAN MODEL</span>"
        border_col = "#ef4444"
    else:
        badge = "<span style='background:#92400e;color:#fde68a;padding:3px 10px;border-radius:999px;font-size:11px;font-weight:700;'>EXPERIMENTAL HIGHER THAN MODEL</span>"
        border_col = "#f59e0b"

    reasons = [
        f"<b>Material behavior basis:</b> {mat_note}",
        "<b>Initial crookedness &amp; load eccentricity:</b> Even small geometric straightness errors or off-center axial loading create bending moments earlier than ideal Euler theory.",
        f"<b>End-condition realism ({bc_label}):</b> Real supports have finite rotational stiffness or slight end slip compared with ideal analytical K values.",
    ]
    if section_type == "Hollow Tube":
        reasons.append("<b>Thin-wall local effects:</b> Thin tube walls can experience local wall distortion/ovalization alongside global column bending.")

    reasons_html = "".join(f"<li style='margin-bottom:4px;'>{r}</li>" for r in reasons)

    html = f"""
    <div style="
        background: linear-gradient(135deg, #0f172a 0%, #1e293b 100%);
        border: 1px solid #334155;
        border-left: 4px solid {border_col};
        border-radius: 8px;
        padding: 14px 18px;
        margin-top: 10px;
        color: #f8fafc;
        font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;
    ">
        <div style="display:flex;justify-content:space-between;align-items:center;margin-bottom:8px;">
            <div style="font-size:13px;font-weight:700;">Model vs. Experiment Comparison &amp; Physical Reasons</div>
            {badge}
        </div>
        <div style="font-size:13px;line-height:1.55;color:#e2e8f0;margin-bottom:8px;">
            Simple model <b>P_model = {res['P_model_N']:.2f} N</b> ({res['Governing']}) vs.
            Experimental <b>P_exp = {res['P_exp_N']:.2f} N</b> &rarr;
            Difference: <b>{diff_pct:+.1f}%</b> (Knockdown P_exp / P_model = <b>{res['Knockdown']:.2f}</b>,
            Slenderness K·L/k = <b>{res['Slenderness']:.1f}</b>).
        </div>
        <div style="font-size:12.5px;color:#cbd5e1;">
            <ul style="margin:4px 0 0 18px;padding:0;line-height:1.45;">{reasons_html}</ul>
        </div>
    </div>
    """
    display(HTML(html))

# ==========================================
# 5. INTERACTIVE ENGLISH UI
# ==========================================
style = {"description_width": "145px"}
layout_w = widgets.Layout(width="335px")

mat_dd = widgets.Dropdown(
    options=list(COLUMN_MATERIALS.keys()),
    value="Paper (Cardstock / A4 Roll)",
    description="Column Material:",
    style=style, layout=layout_w
)

sec_dd = widgets.Dropdown(
    options=["Hollow Tube", "Rectangular Strip", "Solid Square"],
    value="Hollow Tube",
    description="Cross-Section:",
    style=style, layout=layout_w
)

bc_dd = widgets.Dropdown(
    options=list(BOUNDARY_CONDITIONS.keys()),
    value="Pinned-Pinned (K = 1.0)",
    description="End Condition:",
    style=style, layout=layout_w
)

L_sl = widgets.FloatSlider(
    value=250.0, min=80.0, max=600.0, step=10.0,
    description="Length L [mm]:",
    readout_format=".0f", style=style, layout=layout_w
)

d1_sl = widgets.FloatSlider(
    value=20.0, min=4.0, max=80.0, step=1.0,
    description="Outer Dia. D [mm]:",
    readout_format=".1f", style=style, layout=layout_w
)

d2_sl = widgets.FloatSlider(
    value=2.0, min=0.5, max=30.0, step=0.5,
    description="Strip h [mm]:",
    readout_format=".1f", style=style, layout=layout_w
)

wall_sl = widgets.FloatSlider(
    value=0.6, min=0.1, max=6.0, step=0.1,
    description="Tube Wall t [mm]:",
    readout_format=".1f", style=style, layout=layout_w
)

p_exp_sl = widgets.FloatSlider(
    value=18.0, min=0.5, max=500.0, step=0.5,
    description="Measured P_exp [N]:",
    readout_format=".1f", style=style, layout=layout_w
)

preset_btn = widgets.Button(
    description="Set Realistic Test P_exp",
    button_style="info",
    icon="sliders",
    layout=widgets.Layout(width="210px", height="34px")
)

run_btn = widgets.Button(
    description="Compare Model vs. Test",
    button_style="primary",
    icon="bar-chart",
    layout=widgets.Layout(width="210px", height="36px")
)

out = widgets.Output()

def toggle_geom_inputs(*args):
    st = sec_dd.value
    wall_sl.layout.display = "flex" if st == "Hollow Tube" else "none"
    d2_sl.layout.display = "flex" if st == "Rectangular Strip" else "none"
    if st == "Hollow Tube":
        d1_sl.description = "Outer Dia. D [mm]:"
    elif st == "Rectangular Strip":
        d1_sl.description = "Strip Width b [mm]:"
    else:
        d1_sl.description = "Square Side a [mm]:"

sec_dd.observe(toggle_geom_inputs, names="value")
toggle_geom_inputs()

def apply_realistic_p_exp(*args):
    p_exp_sl.value = realistic_test_load_N(
        mat_dd.value, sec_dd.value, L_sl.value,
        d1_sl.value, d2_sl.value, wall_sl.value,
        BOUNDARY_CONDITIONS[bc_dd.value]
    )

def run_buckling_ui(*args):
    with out:
        clear_output(wait=True)
        K_val = BOUNDARY_CONDITIONS[bc_dd.value]
        res = solve_column_buckling(
            mat_dd.value, sec_dd.value, L_sl.value,
            d1_sl.value, d2_sl.value, wall_sl.value,
            K_val, p_exp_sl.value
        )

        plot_buckling_comparison(res, mat_dd.value, sec_dd.value, bc_dd.value)

        summary_tbl = pd.DataFrame([{
            "Geometry": res["Geom_Desc"],
            "Area [mm²]": round(res["A_mm2"], 2),
            "I [mm⁴]": round(res["I_mm4"], 1),
            "K·L/k": round(res["Slenderness"], 1),
            "Euler P_cr [N]": round(res["P_euler_N"], 2),
            "Model P_model [N]": round(res["P_model_N"], 2),
            "Measured P_exp [N]": round(res["P_exp_N"], 2),
            "Diff [%]": round(res["Diff_pct"], 1),
        }])
        display(summary_tbl.style.hide(axis="index"))

        render_comparison_summary(res, mat_dd.value, sec_dd.value, bc_dd.value)

preset_btn.on_click(apply_realistic_p_exp)
run_btn.on_click(run_buckling_ui)

ui = widgets.VBox([
    widgets.HTML("""
        <div style="padding:10px 14px;background:#0f172a;border-radius:8px;border:1px solid #334155;margin-bottom:8px;">
            <div style="font-size:17px;font-weight:700;color:#f8fafc;">Column Buckling Tester — Paper, Balsa &amp; 3D-Printed vs. Simple Model</div>
            <div style="font-size:12px;color:#94a3b8;">Realistic material &amp; knockdown-based test values with interactive comparison</div>
        </div>
    """),
    widgets.HBox([
        widgets.VBox([mat_dd, sec_dd, bc_dd, p_exp_sl], layout=widgets.Layout(margin="0 14px 0 0")),
        widgets.VBox([L_sl, d1_sl, d2_sl, wall_sl], layout=widgets.Layout(margin="0 14px 0 0")),
        widgets.VBox([preset_btn, run_btn], layout=widgets.Layout(justify_content="center", padding="8px", gap="8px"))
    ], layout=widgets.Layout(align_items="flex-start"))
])

display(ui, out)
apply_realistic_p_exp()
run_buckling_ui()

Output()

In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

# ==========================================
# 1. REALISTIC MATERIAL & BOUNDARY DATA
# ==========================================
MATERIALS = {
    "Paper": {
        "E": 2.2e9, "sigma_c": 22e6, "knockdown": 0.72,
        "desc": "Cardstock/rolled paper (effective E ~ 2.2 GPa). Seams, creases, moisture, and load offset typically reduce test load to ~65-80% of Euler load."
    },
    "Balsa Wood": {
        "E": 3.5e9, "sigma_c": 12e6, "knockdown": 0.78,
        "desc": "Lightweight balsa wood (E ~ 3.5 GPa). Grain direction, density variation, and minor off-center load typically give ~70-85% of Euler load."
    },
    "3D-Printed PLA": {
        "E": 2.8e9, "sigma_c": 45e6, "knockdown": 0.82,
        "desc": "3D-printed PLA column (E ~ 2.8 GPa). Layer bonding, infill pattern, and slight imperfections typically give ~75-90% of Euler load."
    },
}

END_CONDITIONS = {
    "Pinned-Pinned (K=1.0)": 1.0,
    "Fixed-Pinned (K=0.7)":  0.7,
    "Fixed-Fixed (K=0.5)":   0.5,
    "Cantilever (K=2.0)":    2.0,
}

# ==========================================
# 2. GEOMETRY & EULER BUCKLING SOLVER
# ==========================================
def get_geometry_and_load(mat_name, sec_type, L_mm, d1_mm, d2_mm, wall_mm, K_val):
    mat = MATERIALS[mat_name]
    L = L_mm * 1e-3
    Le = K_val * L

    if sec_type == "Hollow Tube":
        D_out = d1_mm * 1e-3
        t = min(wall_mm * 1e-3, 0.45 * D_out)
        D_in = max(D_out - 2.0 * t, 0.0)
        A = (np.pi / 4.0) * (D_out**2 - D_in**2)
        I = (np.pi / 64.0) * (D_out**4 - D_in**4)
        geom_info = {"type": "tube", "D_out_mm": d1_mm, "D_in_mm": D_in * 1e3, "t_mm": t * 1e3}
        geom_text = f"Hollow Tube (Ø {d1_mm:.1f} mm, wall {t*1e3:.2f} mm)"
    elif sec_type == "Rectangular Strip":
        b = d1_mm * 1e-3
        h = d2_mm * 1e-3
        A = b * h
        I = (b * h**3) / 12.0
        geom_info = {"type": "strip", "b_mm": d1_mm, "h_mm": d2_mm}
        geom_text = f"Rectangular Strip ({d1_mm:.1f} x {d2_mm:.1f} mm)"
    else:  # "Solid Square"
        a = d1_mm * 1e-3
        A = a**2
        I = (a**4) / 12.0
        geom_info = {"type": "square", "a_mm": d1_mm}
        geom_text = f"Solid Square ({d1_mm:.1f} x {d1_mm:.1f} mm)"

    k_gyr = np.sqrt(I / max(A, 1e-15))
    slenderness = Le / max(k_gyr, 1e-9)

    P_euler = (np.pi**2 * mat["E"] * I) / max(Le**2, 1e-15)
    P_crush = mat["sigma_c"] * A
    P_model = min(P_euler, P_crush)
    mode = "Euler Buckling" if P_euler <= P_crush else "Compression Yield"

    sec_factor = 0.95 if sec_type == "Hollow Tube" else (0.90 if sec_type == "Rectangular Strip" else 0.96)
    P_realistic_exp = float(np.clip(round(P_model * mat["knockdown"] * sec_factor, 1), 0.5, 500.0))

    return {
        "A_mm2": A * 1e6,
        "I_mm4": I * 1e12,
        "k_mm": k_gyr * 1e3,
        "slenderness": slenderness,
        "P_euler": P_euler,
        "P_crush": P_crush,
        "P_model": P_model,
        "P_realistic_exp": P_realistic_exp,
        "mode": mode,
        "geom_info": geom_info,
        "geom_text": geom_text,
    }

# ==========================================
# 3. COMPARISON GRAPHIC + FULL 2D DRAWING
# ==========================================
def draw_comparison_and_2d_column(mat_name, sec_type, bc_name, L_mm, d1_mm, d2_mm, wall_mm, P_exp_N):
    K_val = END_CONDITIONS[bc_name]
    info = get_geometry_and_load(mat_name, sec_type, L_mm, d1_mm, d2_mm, wall_mm, K_val)
    P_model = info["P_model"]
    diff_pct = ((P_exp_N - P_model) / max(P_model, 1e-12)) * 100.0

    plt.style.use("seaborn-v0_8-whitegrid")

    # --- GRAPHIC 1: Load Comparison Bar Chart ---
    fig1, ax_bar = plt.subplots(figsize=(8.5, 3.8))
    categories = ["Simple Model (P_model)", "Experimental Load (P_exp)"]
    values = [P_model, P_exp_N]
    colors = ["#2563eb", "#f97316"]

    bars = ax_bar.bar(categories, values, color=colors, edgecolor="#0f172a", linewidth=1.2, width=0.45)
    ax_bar.set_title(f"Buckling Load Comparison — {mat_name} | {info['geom_text']}", fontsize=11, fontweight="bold")
    ax_bar.set_ylabel("Axial Load [N]", fontsize=9.5)
    top_y = max(values) * 1.22 if max(values) > 0 else 1.0
    ax_bar.set_ylim(0, top_y)

    for bar, val in zip(bars, values):
        ax_bar.text(
            bar.get_x() + bar.get_width() / 2.0,
            val + top_y * 0.025,
            f"{val:.2f} N",
            ha="center", va="bottom", fontsize=9.5, fontweight="bold"
        )

    ax_bar.text(
        0.98, 0.90,
        f"Difference: {diff_pct:+.1f}%\nGoverning Model: {info['mode']}\nSlenderness (K·L/k): {info['slenderness']:.1f}",
        transform=ax_bar.transAxes, ha="right", va="top", fontsize=8.5,
        bbox=dict(boxstyle="round,pad=0.4", facecolor="#f8fafc", edgecolor="#94a3b8")
    )
    plt.tight_layout()
    plt.show()

    # --- GRAPHIC 2: Full 2D Drawing of Selected Column (Elevation + Cross-Section) ---
    fig2, (ax_elev, ax_sec) = plt.subplots(1, 2, figsize=(11.5, 4.4), gridspec_kw={"width_ratios": [1.5, 1.0]})

    # Full 2D elevation along height z in [0, L_mm]
    s = np.linspace(0.0, 1.0, 200)
    z_mm = s * L_mm

    if "Pinned-Pinned" in bc_name:
        w_norm = np.sin(np.pi * s)
    elif "Fixed-Fixed" in bc_name:
        w_norm = 0.5 * (1.0 - np.cos(2.0 * np.pi * s))
    elif "Cantilever" in bc_name:
        w_norm = 1.0 - np.cos(0.5 * np.pi * s)
    else:
        w_norm = np.sin(np.pi * s) - 0.25 * np.sin(2.0 * np.pi * s)
        w_norm = w_norm / max(np.max(np.abs(w_norm)), 1e-9)

    defl_amp = max(0.08 * L_mm, 5.0)
    half_w = max(d1_mm * 0.5, 2.0)

    # Undeformed 2D column outline
    ax_elev.plot([-half_w, -half_w], [0, L_mm], color="#64748b", lw=1.4, ls="--", label="Undeformed Column")
    ax_elev.plot([half_w, half_w], [0, L_mm], color="#64748b", lw=1.4, ls="--")
    ax_elev.plot([0, 0], [0, L_mm], color="#94a3b8", lw=0.9, ls=":")

    # Buckled centerline & walls (2D)
    w_curve = w_norm * defl_amp
    ax_elev.plot(w_curve, z_mm, color="#dc2626", lw=2.5, label="Buckled Centerline")
    ax_elev.plot(w_curve - half_w, z_mm, color="#f97316", lw=1.6, label="Buckled 2D Profile")
    ax_elev.plot(w_curve + half_w, z_mm, color="#f97316", lw=1.6)

    # Top load arrow & bottom base support
    top_x = w_curve[-1]
    ax_elev.annotate("", xy=(top_x, L_mm), xytext=(top_x, L_mm + 0.14 * L_mm),
                     arrowprops=dict(arrowstyle="->", color="#0f172a", lw=2.2))
    ax_elev.text(top_x, L_mm + 0.16 * L_mm, f"P_exp = {P_exp_N:.1f} N", ha="center", va="bottom", fontsize=9, fontweight="bold")
    ax_elev.plot([-1.6 * half_w, 1.6 * half_w], [0, 0], color="#0f172a", lw=3.0)
    ax_elev.text(0, -0.08 * L_mm, f"Base: {bc_name}", ha="center", va="top", fontsize=8.5, fontweight="bold")

    ax_elev.set_title(f"Full 2D Column View (Length L = {L_mm:.0f} mm)", fontsize=10.5, fontweight="bold")
    ax_elev.set_xlabel("Transverse Position / Deflection [mm]", fontsize=9)
    ax_elev.set_ylabel("Column Height z [mm]", fontsize=9)
    x_lim_elev = max(defl_amp * 1.45, half_w * 2.2)
    ax_elev.set_xlim(-x_lim_elev, x_lim_elev)
    ax_elev.set_ylim(-0.14 * L_mm, L_mm * 1.24)
    ax_elev.legend(loc="upper left", fontsize=8, frameon=True, facecolor="white")

    # 2D Cross-Section Drawing
    g = info["geom_info"]
    if g["type"] == "tube":
        r_out = g["D_out_mm"] / 2.0
        r_in = g["D_in_mm"] / 2.0
        c_out = patches.Circle((0, 0), r_out, facecolor="#3b82f6", edgecolor="#0f172a", lw=1.5, alpha=0.88)
        c_in = patches.Circle((0, 0), r_in, facecolor="white", edgecolor="#0f172a", lw=1.2)
        ax_sec.add_patch(c_out)
        ax_sec.add_patch(c_in)
        lim_s = r_out * 1.35
        sec_title = f"2D Cross-Section: Hollow Tube\nØ_out={g['D_out_mm']:.1f} mm, t={g['t_mm']:.2f} mm"
    elif g["type"] == "strip":
        bw, hh = g["b_mm"], g["h_mm"]
        rect = patches.Rectangle((-bw/2, -hh/2), bw, hh, facecolor="#3b82f6", edgecolor="#0f172a", lw=1.5, alpha=0.88)
        ax_sec.add_patch(rect)
        lim_s = max(bw, hh) * 0.85
        sec_title = f"2D Cross-Section: Rectangular Strip\nb={bw:.1f} mm, h={hh:.1f} mm"
    else:
        aa = g["a_mm"]
        sq = patches.Rectangle((-aa/2, -aa/2), aa, aa, facecolor="#3b82f6", edgecolor="#0f172a", lw=1.5, alpha=0.88)
        ax_sec.add_patch(sq)
        lim_s = aa * 0.85
        sec_title = f"2D Cross-Section: Solid Square\na={aa:.1f} mm"

    ax_sec.set_xlim(-lim_s, lim_s)
    ax_sec.set_ylim(-lim_s, lim_s)
    ax_sec.set_aspect("equal")
    ax_sec.set_title(sec_title, fontsize=10, fontweight="bold")
    ax_sec.set_xlabel("x [mm]", fontsize=9)
    ax_sec.set_ylabel("y [mm]", fontsize=9)

    plt.tight_layout()
    plt.show()

    # --- SUMMARY & WHY THEY DIFFER ---
    if abs(diff_pct) <= 15.0:
        status_html = "<span style='color:#34d399;font-weight:700;'>Close match with simple model</span>"
    elif diff_pct < -15.0:
        status_html = "<span style='color:#f87171;font-weight:700;'>Experimental load is lower than simple model</span>"
    else:
        status_html = "<span style='color:#fbbf24;font-weight:700;'>Experimental load is higher than simple model</span>"

    display(HTML(f"""
    <div style="background:#0f172a;border:1px solid #334155;border-left:4px solid #38bdf8;
                border-radius:8px;padding:12px 16px;color:#f8fafc;font-family:sans-serif;margin-top:6px;">
        <div style="font-size:13px;margin-bottom:6px;">
            <b>Result:</b> {status_html} &nbsp;|&nbsp;
            Simple Model: <b>{P_model:.2f} N</b> &nbsp;|&nbsp;
            Experimental: <b>{P_exp_N:.2f} N</b> &nbsp;|&nbsp;
            Diff: <b>{diff_pct:+.1f}%</b>
        </div>
        <div style="font-size:12.5px;color:#cbd5e1;line-height:1.5;">
            <b>Why they differ:</b> {MATERIALS[mat_name]['desc']}
            Ideal Euler theory assumes a perfectly straight column, true centered axial load, and ideal end supports ({bc_name}),
            whereas real tests include slight initial crookedness, load eccentricity, and fixture compliance.
        </div>
    </div>
    """))

# ==========================================
# 4. CLEAR, SELECTABLE CONTROLS IN COLAB
# ==========================================
w_mat = widgets.Dropdown(options=list(MATERIALS.keys()), value="Paper", description="Material:")
w_sec = widgets.Dropdown(options=["Hollow Tube", "Rectangular Strip", "Solid Square"], value="Hollow Tube", description="Section:")
w_bc  = widgets.Dropdown(options=list(END_CONDITIONS.keys()), value="Pinned-Pinned (K=1.0)", description="End Support:")

w_L    = widgets.FloatSlider(value=250.0, min=80.0, max=600.0, step=10.0, description="Length L [mm]:")
w_d1   = widgets.FloatSlider(value=20.0,  min=4.0,  max=80.0,  step=1.0,  description="Dia / Width [mm]:")
w_d2   = widgets.FloatSlider(value=2.0,   min=0.5,  max=25.0,  step=0.5,  description="Strip h [mm]:")
w_wall = widgets.FloatSlider(value=0.6,   min=0.1,  max=5.0,   step=0.1,  description="Tube Wall [mm]:")
w_pexp = widgets.FloatSlider(value=12.0,  min=0.5,  max=300.0, step=0.5,  description="Exp Load [N]:")

btn_real = widgets.Button(description="Fill Realistic P_exp", button_style="info")
btn_run  = widgets.Button(description="Update Chart & 2D Drawing", button_style="primary")

out_box = widgets.Output()

def sync_section_fields(*args):
    s = w_sec.value
    w_wall.layout.display = "" if s == "Hollow Tube" else "none"
    w_d2.layout.display = "" if s == "Rectangular Strip" else "none"
    w_d1.description = "Outer Ø [mm]:" if s == "Hollow Tube" else ("Width b [mm]:" if s == "Rectangular Strip" else "Side a [mm]:")

w_sec.observe(sync_section_fields, names="value")
sync_section_fields()

def fill_realistic(*args):
    info = get_geometry_and_load(
        w_mat.value, w_sec.value, w_L.value, w_d1.value, w_d2.value, w_wall.value,
        END_CONDITIONS[w_bc.value]
    )
    w_pexp.value = info["P_realistic_exp"]
    render_view()

def render_view(*args):
    with out_box:
        clear_output(wait=True)
        draw_comparison_and_2d_column(
            w_mat.value, w_sec.value, w_bc.value,
            w_L.value, w_d1.value, w_d2.value, w_wall.value, w_pexp.value
        )

btn_real.on_click(fill_realistic)
btn_run.on_click(render_view)

for ctrl in [w_mat, w_sec, w_bc, w_L, w_d1, w_d2, w_wall, w_pexp]:
    ctrl.observe(render_view, names="value")

panel = widgets.VBox([
    widgets.HTML("<h3>10. Column Buckling — Model vs. Test & Full 2D Drawing</h3>"),
    widgets.HBox([widgets.VBox([w_mat, w_sec, w_bc, w_pexp]), widgets.VBox([w_L, w_d1, w_d2, w_wall])]),
    widgets.HBox([btn_real, btn_run], layout=widgets.Layout(margin="8px 0"))
])

display(panel, out_box)
fill_realistic()

Output()

In [4]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

# ==========================================
# 1. REALISTIC MATERIAL & BOUNDARY DATA
# ==========================================
MATERIALS = {
    "Paper": {
        "E": 2.2e9, "sigma_c": 22e6, "knockdown": 0.72,
        "desc": "Cardstock/rolled paper (effective E ~ 2.2 GPa). Seams, creases, moisture, and load offset typically reduce test load to ~65-80% of Euler load."
    },
    "Balsa Wood": {
        "E": 3.5e9, "sigma_c": 12e6, "knockdown": 0.78,
        "desc": "Lightweight balsa wood (E ~ 3.5 GPa). Grain direction, density variation, and minor off-center load typically give ~70-85% of Euler load."
    },
    "3D-Printed PLA": {
        "E": 2.8e9, "sigma_c": 45e6, "knockdown": 0.82,
        "desc": "3D-printed PLA column (E ~ 2.8 GPa). Layer bonding, infill pattern, and slight imperfections typically give ~75-90% of Euler load."
    },
}

END_CONDITIONS = {
    "Pinned-Pinned (K=1.0)": 1.0,
    "Fixed-Pinned (K=0.7)":  0.7,
    "Fixed-Fixed (K=0.5)":   0.5,
    "Cantilever (K=2.0)":    2.0,
}

# ==========================================
# 2. GEOMETRY & EULER BUCKLING SOLVER
# ==========================================
def get_geometry_and_load(mat_name, sec_type, L_mm, d1_mm, d2_mm, wall_mm, K_val):
    mat = MATERIALS[mat_name]
    L = L_mm * 1e-3
    Le = K_val * L

    if sec_type == "Hollow Tube":
        D_out = d1_mm * 1e-3
        t = min(wall_mm * 1e-3, 0.45 * D_out)
        D_in = max(D_out - 2.0 * t, 0.0)
        A = (np.pi / 4.0) * (D_out**2 - D_in**2)
        I = (np.pi / 64.0) * (D_out**4 - D_in**4)
        geom_info = {"type": "tube", "D_out_mm": d1_mm, "D_in_mm": D_in * 1e3, "t_mm": t * 1e3}
        geom_text = f"Hollow Tube (Ø {d1_mm:.1f} mm, wall {t*1e3:.2f} mm)"
    elif sec_type == "Rectangular Strip":
        b = d1_mm * 1e-3
        h = d2_mm * 1e-3
        A = b * h
        I = (b * h**3) / 12.0
        geom_info = {"type": "strip", "b_mm": d1_mm, "h_mm": d2_mm}
        geom_text = f"Rectangular Strip ({d1_mm:.1f} x {d2_mm:.1f} mm)"
    else:  # "Solid Square"
        a = d1_mm * 1e-3
        A = a**2
        I = (a**4) / 12.0
        geom_info = {"type": "square", "a_mm": d1_mm}
        geom_text = f"Solid Square ({d1_mm:.1f} x {d1_mm:.1f} mm)"

    k_gyr = np.sqrt(I / max(A, 1e-15))
    slenderness = Le / max(k_gyr, 1e-9)

    P_euler = (np.pi**2 * mat["E"] * I) / max(Le**2, 1e-15)
    P_crush = mat["sigma_c"] * A
    P_model = min(P_euler, P_crush)
    mode = "Euler Buckling" if P_euler <= P_crush else "Compression Yield"

    sec_factor = 0.95 if sec_type == "Hollow Tube" else (0.90 if sec_type == "Rectangular Strip" else 0.96)
    P_realistic_exp = float(np.clip(round(P_model * mat["knockdown"] * sec_factor, 1), 0.5, 300.0))

    return {
        "A_mm2": A * 1e6,
        "I_mm4": I * 1e12,
        "k_mm": k_gyr * 1e3,
        "slenderness": slenderness,
        "P_euler": P_euler,
        "P_crush": P_crush,
        "P_model": P_model,
        "P_realistic_exp": P_realistic_exp,
        "mode": mode,
        "geom_info": geom_info,
        "geom_text": geom_text,
    }

# ==========================================
# 3. COMPARISON GRAPHIC + FULL 2D DRAWING
# ==========================================
def draw_comparison_and_2d_column(mat_name, sec_type, bc_name, L_mm, d1_mm, d2_mm, wall_mm, P_exp_N):
    K_val = END_CONDITIONS[bc_name]
    info = get_geometry_and_load(mat_name, sec_type, L_mm, d1_mm, d2_mm, wall_mm, K_val)
    P_model = info["P_model"]
    diff_pct = ((P_exp_N - P_model) / max(P_model, 1e-12)) * 100.0

    plt.style.use("seaborn-v0_8-whitegrid")

    # --- GRAPHIC 1: Load Comparison Bar Chart ---
    fig1, ax_bar = plt.subplots(figsize=(9.2, 3.8))
    categories = ["Simple Model (P_model)", "Experimental Load (P_exp)"]
    values = [P_model, P_exp_N]
    colors = ["#3b82f6", "#f97316"]

    bars = ax_bar.bar(categories, values, color=colors, edgecolor="#0f172a", linewidth=1.2, width=0.42, alpha=0.93)
    ax_bar.set_title(f"Buckling Load Comparison — {mat_name} | {info['geom_text']}", fontsize=11, fontweight="bold", pad=10)
    ax_bar.set_ylabel("Axial Load [N]", fontsize=9.5)
    top_y = max(values) * 1.24 if max(values) > 0 else 1.0
    ax_bar.set_ylim(0, top_y)

    for bar, val in zip(bars, values):
        ax_bar.text(
            bar.get_x() + bar.get_width() / 2.0,
            val + top_y * 0.025,
            f"{val:.2f} N",
            ha="center", va="bottom", fontsize=9.5, fontweight="bold", color="#0f172a"
        )

    ax_bar.text(
        0.98, 0.90,
        f"Difference: {diff_pct:+.1f}%\nGoverning Model: {info['mode']}\nSlenderness (K·L/k): {info['slenderness']:.1f}",
        transform=ax_bar.transAxes, ha="right", va="top", fontsize=8.5,
        bbox=dict(boxstyle="round,pad=0.45", facecolor="#f8fafc", edgecolor="#cbd5e1")
    )
    plt.tight_layout()
    plt.show()

    # --- GRAPHIC 2: Full 2D Drawing of Selected Column (Elevation + Cross-Section) ---
    fig2, (ax_elev, ax_sec) = plt.subplots(1, 2, figsize=(11.8, 4.5), gridspec_kw={"width_ratios": [1.5, 1.0]})

    s = np.linspace(0.0, 1.0, 200)
    z_mm = s * L_mm

    if "Pinned-Pinned" in bc_name:
        w_norm = np.sin(np.pi * s)
    elif "Fixed-Fixed" in bc_name:
        w_norm = 0.5 * (1.0 - np.cos(2.0 * np.pi * s))
    elif "Cantilever" in bc_name:
        w_norm = 1.0 - np.cos(0.5 * np.pi * s)
    else:
        w_norm = np.sin(np.pi * s) - 0.25 * np.sin(2.0 * np.pi * s)
        w_norm = w_norm / max(np.max(np.abs(w_norm)), 1e-9)

    defl_amp = max(0.08 * L_mm, 5.0)
    half_w = max(d1_mm * 0.5, 2.0)

    ax_elev.plot([-half_w, -half_w], [0, L_mm], color="#64748b", lw=1.4, ls="--", label="Undeformed Column")
    ax_elev.plot([half_w, half_w], [0, L_mm], color="#64748b", lw=1.4, ls="--")
    ax_elev.plot([0, 0], [0, L_mm], color="#94a3b8", lw=0.9, ls=":")

    w_curve = w_norm * defl_amp
    ax_elev.plot(w_curve, z_mm, color="#dc2626", lw=2.5, label="Buckled Centerline")
    ax_elev.plot(w_curve - half_w, z_mm, color="#f97316", lw=1.6, label="Buckled 2D Profile")
    ax_elev.plot(w_curve + half_w, z_mm, color="#f97316", lw=1.6)

    top_x = w_curve[-1]
    ax_elev.annotate("", xy=(top_x, L_mm), xytext=(top_x, L_mm + 0.14 * L_mm),
                     arrowprops=dict(arrowstyle="->", color="#0f172a", lw=2.2))
    ax_elev.text(top_x, L_mm + 0.16 * L_mm, f"P_exp = {P_exp_N:.1f} N", ha="center", va="bottom", fontsize=9, fontweight="bold")
    ax_elev.plot([-1.6 * half_w, 1.6 * half_w], [0, 0], color="#0f172a", lw=3.0)
    ax_elev.text(0, -0.08 * L_mm, f"Base: {bc_name}", ha="center", va="top", fontsize=8.5, fontweight="bold")

    ax_elev.set_title(f"Full 2D Column View (Length L = {L_mm:.0f} mm)", fontsize=10.5, fontweight="bold", pad=10)
    ax_elev.set_xlabel("Transverse Position / Deflection [mm]", fontsize=9)
    ax_elev.set_ylabel("Column Height z [mm]", fontsize=9)
    x_lim_elev = max(defl_amp * 1.45, half_w * 2.2)
    ax_elev.set_xlim(-x_lim_elev, x_lim_elev)
    ax_elev.set_ylim(-0.14 * L_mm, L_mm * 1.24)
    ax_elev.legend(loc="upper left", fontsize=8, frameon=True, facecolor="white", edgecolor="#cbd5e1")

    g = info["geom_info"]
    if g["type"] == "tube":
        r_out = g["D_out_mm"] / 2.0
        r_in = g["D_in_mm"] / 2.0
        c_out = patches.Circle((0, 0), r_out, facecolor="#3b82f6", edgecolor="#0f172a", lw=1.5, alpha=0.9)
        c_in = patches.Circle((0, 0), r_in, facecolor="white", edgecolor="#0f172a", lw=1.2)
        ax_sec.add_patch(c_out)
        ax_sec.add_patch(c_in)
        lim_s = r_out * 1.35
        sec_title = f"2D Cross-Section: Hollow Tube\nØ_out={g['D_out_mm']:.1f} mm, t={g['t_mm']:.2f} mm"
    elif g["type"] == "strip":
        bw, hh = g["b_mm"], g["h_mm"]
        rect = patches.Rectangle((-bw/2, -hh/2), bw, hh, facecolor="#3b82f6", edgecolor="#0f172a", lw=1.5, alpha=0.9)
        ax_sec.add_patch(rect)
        lim_s = max(bw, hh) * 0.85
        sec_title = f"2D Cross-Section: Rectangular Strip\nb={bw:.1f} mm, h={hh:.1f} mm"
    else:
        aa = g["a_mm"]
        sq = patches.Rectangle((-aa/2, -aa/2), aa, aa, facecolor="#3b82f6", edgecolor="#0f172a", lw=1.5, alpha=0.9)
        ax_sec.add_patch(sq)
        lim_s = aa * 0.85
        sec_title = f"2D Cross-Section: Solid Square\na={aa:.1f} mm"

    ax_sec.set_xlim(-lim_s, lim_s)
    ax_sec.set_ylim(-lim_s, lim_s)
    ax_sec.set_aspect("equal")
    ax_sec.set_title(sec_title, fontsize=10, fontweight="bold", pad=10)
    ax_sec.set_xlabel("x [mm]", fontsize=9)
    ax_sec.set_ylabel("y [mm]", fontsize=9)

    plt.tight_layout()
    plt.show()

    # --- SUMMARY & WHY THEY DIFFER ---
    if abs(diff_pct) <= 15.0:
        status_html = "<span style='color:#34d399;font-weight:700;'>Close match with simple model</span>"
        border_col = "#10b981"
    elif diff_pct < -15.0:
        status_html = "<span style='color:#f87171;font-weight:700;'>Experimental load is lower than simple model</span>"
        border_col = "#ef4444"
    else:
        status_html = "<span style='color:#fbbf24;font-weight:700;'>Experimental load is higher than simple model</span>"
        border_col = "#f59e0b"

    display(HTML(f"""
    <div style="
        background: linear-gradient(135deg, #0f172a 0%, #1e293b 100%);
        border: 1px solid #334155;
        border-left: 4px solid {border_col};
        border-radius: 8px;
        padding: 13px 17px;
        color: #f8fafc;
        font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;
        margin-top: 8px;
    ">
        <div style="font-size:13px;margin-bottom:6px;">
            <b>Result:</b> {status_html} &nbsp;|&nbsp;
            Simple Model: <b>{P_model:.2f} N</b> &nbsp;|&nbsp;
            Experimental: <b>{P_exp_N:.2f} N</b> &nbsp;|&nbsp;
            Diff: <b>{diff_pct:+.1f}%</b>
        </div>
        <div style="font-size:12.5px;color:#cbd5e1;line-height:1.5;">
            <b>Why they differ:</b> {MATERIALS[mat_name]['desc']}
            Ideal Euler theory assumes a perfectly straight column, true centered axial load, and ideal end supports ({bc_name}),
            whereas real tests include slight initial crookedness, load eccentricity, and fixture compliance.
        </div>
    </div>
    """))

# ==========================================
# 4. AESTHETIC & CLEAN ENGLISH UI
# ==========================================
lbl_style = {"description_width": "145px"}
ctrl_layout = widgets.Layout(width="340px")

w_mat = widgets.Dropdown(options=list(MATERIALS.keys()), value="3D-Printed PLA", description="Material:", style=lbl_style, layout=ctrl_layout)
w_sec = widgets.Dropdown(options=["Hollow Tube", "Rectangular Strip", "Solid Square"], value="Hollow Tube", description="Section:", style=lbl_style, layout=ctrl_layout)
w_bc  = widgets.Dropdown(options=list(END_CONDITIONS.keys()), value="Pinned-Pinned (K=1.0)", description="End Support:", style=lbl_style, layout=ctrl_layout)
w_pexp = widgets.FloatSlider(value=46.5, min=0.5, max=300.0, step=0.5, description="Exp Load [N]:", readout_format=".1f", style=lbl_style, layout=ctrl_layout)

w_L    = widgets.FloatSlider(value=420.0, min=80.0, max=600.0, step=10.0, description="Length L [mm]:", readout_format=".0f", style=lbl_style, layout=ctrl_layout)
w_d1   = widgets.FloatSlider(value=10.0,  min=4.0,  max=80.0,  step=0.5,  description="Outer Ø [mm]:", readout_format=".1f", style=lbl_style, layout=ctrl_layout)
w_d2   = widgets.FloatSlider(value=2.0,   min=0.5,  max=25.0,  step=0.5,  description="Strip h [mm]:", readout_format=".1f", style=lbl_style, layout=ctrl_layout)
w_wall = widgets.FloatSlider(value=0.9,   min=0.1,  max=5.0,   step=0.1,  description="Tube Wall [mm]:", readout_format=".1f", style=lbl_style, layout=ctrl_layout)

btn_real = widgets.Button(
    description="Fill Realistic P_exp",
    button_style="info",
    icon="sliders",
    layout=widgets.Layout(width="210px", height="36px")
)
btn_run  = widgets.Button(
    description="Update Chart & 2D Drawing",
    button_style="primary",
    icon="refresh",
    layout=widgets.Layout(width="230px", height="36px")
)

out_box = widgets.Output()

def sync_section_fields(*args):
    s = w_sec.value
    w_wall.layout.display = "flex" if s == "Hollow Tube" else "none"
    w_d2.layout.display = "flex" if s == "Rectangular Strip" else "none"
    if s == "Hollow Tube":
        w_d1.description = "Outer Ø [mm]:"
    elif s == "Rectangular Strip":
        w_d1.description = "Strip Width b [mm]:"
    else:
        w_d1.description = "Square Side a [mm]:"

w_sec.observe(sync_section_fields, names="value")
sync_section_fields()

def fill_realistic(*args):
    info = get_geometry_and_load(
        w_mat.value, w_sec.value, w_L.value, w_d1.value, w_d2.value, w_wall.value,
        END_CONDITIONS[w_bc.value]
    )
    w_pexp.value = info["P_realistic_exp"]
    render_view()

def render_view(*args):
    with out_box:
        clear_output(wait=True)
        draw_comparison_and_2d_column(
            w_mat.value, w_sec.value, w_bc.value,
            w_L.value, w_d1.value, w_d2.value, w_wall.value, w_pexp.value
        )

btn_real.on_click(fill_realistic)
btn_run.on_click(render_view)

for ctrl in [w_mat, w_sec, w_bc, w_L, w_d1, w_d2, w_wall, w_pexp]:
    ctrl.observe(render_view, names="value")

header_card = widgets.HTML("""
    <div style="
        background: linear-gradient(135deg, #0f172a 0%, #1e293b 100%);
        border: 1px solid #334155;
        border-left: 4px solid #38bdf8;
        border-radius: 8px;
        padding: 12px 16px;
        margin-bottom: 10px;
        font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;
    ">
        <div style="font-size:17px;font-weight:700;color:#f8fafc;">Column Buckling — Model vs. Test &amp; Full 2D Drawing</div>
        <div style="font-size:12px;color:#94a3b8;margin-top:2px;">
            Compare Euler buckling load against experimental load with full 2D column &amp; cross-section geometry
        </div>
    </div>
""")

controls_card = widgets.VBox([
    widgets.HBox([
        widgets.VBox([w_mat, w_sec, w_bc, w_pexp], layout=widgets.Layout(margin="0 20px 0 0", gap="6px")),
        widgets.VBox([w_L, w_d1, w_d2, w_wall], layout=widgets.Layout(margin="0 20px 0 0", gap="6px")),
        widgets.VBox([btn_real, btn_run], layout=widgets.Layout(justify_content="center", gap="10px", padding="12px 0 0 0"))
    ], layout=widgets.Layout(align_items="flex-start"))
], layout=widgets.Layout(
    padding="14px 16px",
    border="1px solid #334155",
    border_radius="8px",
    background_color="#1e293b",
    margin="0 0 8px 0"
))

display(header_card, controls_card, out_box)
fill_realistic()

HTML(value='\n    <div style="\n        background: linear-gradient(135deg, #0f172a 0%, #1e293b 100%);\n      …

Output()

In [5]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

# ==========================================
# 1. REALISTIC MATERIAL & BOUNDARY DATA
# ==========================================
MATERIALS = {
    "Paper": {
        "E": 2.2e9, "sigma_c": 22e6, "knockdown": 0.72,
        "desc": "Cardstock/rolled paper (effective E ~ 2.2 GPa). Seams, creases, moisture, and load offset typically reduce test load to ~65-80% of Euler load."
    },
    "Balsa Wood": {
        "E": 3.5e9, "sigma_c": 12e6, "knockdown": 0.78,
        "desc": "Lightweight balsa wood (E ~ 3.5 GPa). Grain direction, density variation, and minor off-center load typically give ~70-85% of Euler load."
    },
    "3D-Printed PLA": {
        "E": 2.8e9, "sigma_c": 45e6, "knockdown": 0.82,
        "desc": "3D-printed PLA column (E ~ 2.8 GPa). Layer bonding, infill pattern, and slight imperfections typically give ~75-90% of Euler load."
    },
}

END_CONDITIONS = {
    "Pinned-Pinned (K=1.0)": 1.0,
    "Fixed-Pinned (K=0.7)":  0.7,
    "Fixed-Fixed (K=0.5)":   0.5,
    "Cantilever (K=2.0)":    2.0,
}

# ==========================================
# 2. GEOMETRY & EULER BUCKLING SOLVER
# ==========================================
def get_geometry_and_load(mat_name, sec_type, L_mm, d1_mm, d2_mm, wall_mm, K_val):
    mat = MATERIALS[mat_name]
    L = L_mm * 1e-3
    Le = K_val * L

    if sec_type == "Hollow Tube":
        D_out = d1_mm * 1e-3
        t = min(wall_mm * 1e-3, 0.45 * D_out)
        D_in = max(D_out - 2.0 * t, 0.0)
        A = (np.pi / 4.0) * (D_out**2 - D_in**2)
        I = (np.pi / 64.0) * (D_out**4 - D_in**4)
        geom_info = {"type": "tube", "D_out_mm": d1_mm, "D_in_mm": D_in * 1e3, "t_mm": t * 1e3}
        geom_text = f"Hollow Tube (Ø {d1_mm:.1f} mm, wall {t*1e3:.2f} mm)"
    elif sec_type == "Rectangular Strip":
        b = d1_mm * 1e-3
        h = d2_mm * 1e-3
        A = b * h
        I = (b * h**3) / 12.0
        geom_info = {"type": "strip", "b_mm": d1_mm, "h_mm": d2_mm}
        geom_text = f"Rectangular Strip ({d1_mm:.1f} x {d2_mm:.1f} mm)"
    else:  # "Solid Square"
        a = d1_mm * 1e-3
        A = a**2
        I = (a**4) / 12.0
        geom_info = {"type": "square", "a_mm": d1_mm}
        geom_text = f"Solid Square ({d1_mm:.1f} x {d1_mm:.1f} mm)"

    k_gyr = np.sqrt(I / max(A, 1e-15))
    slenderness = Le / max(k_gyr, 1e-9)

    P_euler = (np.pi**2 * mat["E"] * I) / max(Le**2, 1e-15)
    P_crush = mat["sigma_c"] * A
    P_model = min(P_euler, P_crush)
    mode = "Euler Buckling" if P_euler <= P_crush else "Compression Yield"

    sec_factor = 0.95 if sec_type == "Hollow Tube" else (0.90 if sec_type == "Rectangular Strip" else 0.96)
    P_realistic_exp = float(np.clip(round(P_model * mat["knockdown"] * sec_factor, 1), 0.5, 300.0))

    return {
        "A_mm2": A * 1e6,
        "I_mm4": I * 1e12,
        "k_mm": k_gyr * 1e3,
        "slenderness": slenderness,
        "P_euler": P_euler,
        "P_crush": P_crush,
        "P_model": P_model,
        "P_realistic_exp": P_realistic_exp,
        "mode": mode,
        "geom_info": geom_info,
        "geom_text": geom_text,
    }

# ==========================================
# 3. FULL-WIDTH COMPARISON GRAPHIC & 2D DRAWING
# ==========================================
def draw_comparison_and_2d_column(mat_name, sec_type, bc_name, L_mm, d1_mm, d2_mm, wall_mm, P_exp_N):
    K_val = END_CONDITIONS[bc_name]
    info = get_geometry_and_load(mat_name, sec_type, L_mm, d1_mm, d2_mm, wall_mm, K_val)
    P_model = info["P_model"]
    diff_pct = ((P_exp_N - P_model) / max(P_model, 1e-12)) * 100.0

    plt.style.use("seaborn-v0_8-whitegrid")

    # --- FULL-WIDTH GRAPHIC 1: Load Comparison Bar Chart ---
    fig1, ax_bar = plt.subplots(figsize=(14.5, 3.9))
    categories = ["Simple Model (P_model)", "Experimental Load (P_exp)"]
    values = [P_model, P_exp_N]
    colors = ["#3b82f6", "#f97316"]

    bars = ax_bar.bar(categories, values, color=colors, edgecolor="#0f172a", linewidth=1.2, width=0.34, alpha=0.94)
    ax_bar.set_title(f"Buckling Load Comparison — {mat_name} | {info['geom_text']}", fontsize=11.5, fontweight="bold", pad=12)
    ax_bar.set_ylabel("Axial Load [N]", fontsize=10)
    top_y = max(values) * 1.25 if max(values) > 0 else 1.0
    ax_bar.set_ylim(0, top_y)

    for bar, val in zip(bars, values):
        ax_bar.text(
            bar.get_x() + bar.get_width() / 2.0,
            val + top_y * 0.025,
            f"{val:.2f} N",
            ha="center", va="bottom", fontsize=10, fontweight="bold", color="#0f172a"
        )

    ax_bar.text(
        0.99, 0.90,
        f"Difference: {diff_pct:+.1f}%\nGoverning Model: {info['mode']}\nSlenderness (K·L/k): {info['slenderness']:.1f}",
        transform=ax_bar.transAxes, ha="right", va="top", fontsize=9,
        bbox=dict(boxstyle="round,pad=0.45", facecolor="#f8fafc", edgecolor="#cbd5e1")
    )
    plt.tight_layout()
    plt.show()

    # --- FULL-WIDTH GRAPHIC 2: Full 2D Drawing of Selected Column ---
    fig2, (ax_elev, ax_sec) = plt.subplots(1, 2, figsize=(14.5, 4.8), gridspec_kw={"width_ratios": [1.65, 1.0]})

    s = np.linspace(0.0, 1.0, 200)
    z_mm = s * L_mm

    if "Pinned-Pinned" in bc_name:
        w_norm = np.sin(np.pi * s)
    elif "Fixed-Fixed" in bc_name:
        w_norm = 0.5 * (1.0 - np.cos(2.0 * np.pi * s))
    elif "Cantilever" in bc_name:
        w_norm = 1.0 - np.cos(0.5 * np.pi * s)
    else:
        w_norm = np.sin(np.pi * s) - 0.25 * np.sin(2.0 * np.pi * s)
        w_norm = w_norm / max(np.max(np.abs(w_norm)), 1e-9)

    defl_amp = max(0.08 * L_mm, 5.0)
    half_w = max(d1_mm * 0.5, 2.0)

    ax_elev.plot([-half_w, -half_w], [0, L_mm], color="#64748b", lw=1.5, ls="--", label="Undeformed Column")
    ax_elev.plot([half_w, half_w], [0, L_mm], color="#64748b", lw=1.5, ls="--")
    ax_elev.plot([0, 0], [0, L_mm], color="#94a3b8", lw=1.0, ls=":")

    w_curve = w_norm * defl_amp
    ax_elev.plot(w_curve, z_mm, color="#dc2626", lw=2.6, label="Buckled Centerline")
    ax_elev.plot(w_curve - half_w, z_mm, color="#f97316", lw=1.8, label="Buckled 2D Profile")
    ax_elev.plot(w_curve + half_w, z_mm, color="#f97316", lw=1.8)

    top_x = w_curve[-1]
    ax_elev.annotate("", xy=(top_x, L_mm), xytext=(top_x, L_mm + 0.14 * L_mm),
                     arrowprops=dict(arrowstyle="->", color="#0f172a", lw=2.2))
    ax_elev.text(top_x, L_mm + 0.16 * L_mm, f"P_exp = {P_exp_N:.1f} N", ha="center", va="bottom", fontsize=9.5, fontweight="bold")
    ax_elev.plot([-1.6 * half_w, 1.6 * half_w], [0, 0], color="#0f172a", lw=3.2)
    ax_elev.text(0, -0.08 * L_mm, f"Base: {bc_name}", ha="center", va="top", fontsize=9, fontweight="bold")

    ax_elev.set_title(f"Full 2D Column View (Length L = {L_mm:.0f} mm)", fontsize=11, fontweight="bold", pad=12)
    ax_elev.set_xlabel("Transverse Position / Deflection [mm]", fontsize=9.5)
    ax_elev.set_ylabel("Column Height z [mm]", fontsize=9.5)
    x_lim_elev = max(defl_amp * 1.45, half_w * 2.2)
    ax_elev.set_xlim(-x_lim_elev, x_lim_elev)
    ax_elev.set_ylim(-0.14 * L_mm, L_mm * 1.25)
    ax_elev.legend(loc="upper left", fontsize=8.5, frameon=True, facecolor="white", edgecolor="#cbd5e1")

    g = info["geom_info"]
    if g["type"] == "tube":
        r_out = g["D_out_mm"] / 2.0
        r_in = g["D_in_mm"] / 2.0
        c_out = patches.Circle((0, 0), r_out, facecolor="#3b82f6", edgecolor="#0f172a", lw=1.6, alpha=0.92)
        c_in = patches.Circle((0, 0), r_in, facecolor="white", edgecolor="#0f172a", lw=1.3)
        ax_sec.add_patch(c_out)
        ax_sec.add_patch(c_in)
        lim_s = r_out * 1.35
        sec_title = f"2D Cross-Section: Hollow Tube\nØ_out={g['D_out_mm']:.1f} mm, t={g['t_mm']:.2f} mm"
    elif g["type"] == "strip":
        bw, hh = g["b_mm"], g["h_mm"]
        rect = patches.Rectangle((-bw/2, -hh/2), bw, hh, facecolor="#3b82f6", edgecolor="#0f172a", lw=1.6, alpha=0.92)
        ax_sec.add_patch(rect)
        lim_s = max(bw, hh) * 0.85
        sec_title = f"2D Cross-Section: Rectangular Strip\nb={bw:.1f} mm, h={hh:.1f} mm"
    else:
        aa = g["a_mm"]
        sq = patches.Rectangle((-aa/2, -aa/2), aa, aa, facecolor="#3b82f6", edgecolor="#0f172a", lw=1.6, alpha=0.92)
        ax_sec.add_patch(sq)
        lim_s = aa * 0.85
        sec_title = f"2D Cross-Section: Solid Square\na={aa:.1f} mm"

    ax_sec.set_xlim(-lim_s, lim_s)
    ax_sec.set_ylim(-lim_s, lim_s)
    ax_sec.set_aspect("equal")
    ax_sec.set_title(sec_title, fontsize=10.5, fontweight="bold", pad=12)
    ax_sec.set_xlabel("x [mm]", fontsize=9.5)
    ax_sec.set_ylabel("y [mm]", fontsize=9.5)

    plt.tight_layout()
    plt.show()

    # --- FULL-WIDTH SUMMARY CARD ---
    if abs(diff_pct) <= 15.0:
        status_html = "<span style='color:#34d399;font-weight:700;'>Close match with simple model</span>"
        border_col = "#10b981"
    elif diff_pct < -15.0:
        status_html = "<span style='color:#f87171;font-weight:700;'>Experimental load is lower than simple model</span>"
        border_col = "#ef4444"
    else:
        status_html = "<span style='color:#fbbf24;font-weight:700;'>Experimental load is higher than simple model</span>"
        border_col = "#f59e0b"

    display(HTML(f"""
    <div style="
        width: 100%;
        box-sizing: border-box;
        background: linear-gradient(135deg, #0f172a 0%, #1e293b 100%);
        border: 1px solid #334155;
        border-left: 4px solid {border_col};
        border-radius: 8px;
        padding: 14px 18px;
        color: #f8fafc;
        font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;
        margin-top: 8px;
    ">
        <div style="font-size:13.5px;margin-bottom:6px;">
            <b>Result:</b> {status_html} &nbsp;|&nbsp;
            Simple Model: <b>{P_model:.2f} N</b> &nbsp;|&nbsp;
            Experimental: <b>{P_exp_N:.2f} N</b> &nbsp;|&nbsp;
            Diff: <b>{diff_pct:+.1f}%</b>
        </div>
        <div style="font-size:12.5px;color:#cbd5e1;line-height:1.55;">
            <b>Why they differ:</b> {MATERIALS[mat_name]['desc']}
            Ideal Euler theory assumes a perfectly straight column, true centered axial load, and ideal end supports ({bc_name}),
            whereas real tests include slight initial crookedness, load eccentricity, and fixture compliance.
        </div>
    </div>
    """))

# ==========================================
# 4. FULL-WIDTH AESTHETIC ENGLISH UI
# ==========================================
lbl_style = {"description_width": "150px"}
ctrl_layout = widgets.Layout(width="380px")

w_mat  = widgets.Dropdown(options=list(MATERIALS.keys()), value="3D-Printed PLA", description="Material:", style=lbl_style, layout=ctrl_layout)
w_sec  = widgets.Dropdown(options=["Hollow Tube", "Rectangular Strip", "Solid Square"], value="Hollow Tube", description="Section:", style=lbl_style, layout=ctrl_layout)
w_bc   = widgets.Dropdown(options=list(END_CONDITIONS.keys()), value="Pinned-Pinned (K=1.0)", description="End Support:", style=lbl_style, layout=ctrl_layout)
w_pexp = widgets.FloatSlider(value=46.5, min=0.5, max=300.0, step=0.5, description="Exp Load [N]:", readout_format=".1f", style=lbl_style, layout=ctrl_layout)

w_L    = widgets.FloatSlider(value=420.0, min=80.0, max=600.0, step=10.0, description="Length L [mm]:", readout_format=".0f", style=lbl_style, layout=ctrl_layout)
w_d1   = widgets.FloatSlider(value=10.0,  min=4.0,  max=80.0,  step=0.5,  description="Outer Ø [mm]:", readout_format=".1f", style=lbl_style, layout=ctrl_layout)
w_d2   = widgets.FloatSlider(value=2.0,   min=0.5,  max=25.0,  step=0.5,  description="Strip h [mm]:", readout_format=".1f", style=lbl_style, layout=ctrl_layout)
w_wall = widgets.FloatSlider(value=0.9,   min=0.1,  max=5.0,   step=0.1,  description="Tube Wall [mm]:", readout_format=".1f", style=lbl_style, layout=ctrl_layout)

btn_real = widgets.Button(
    description="Fill Realistic P_exp",
    button_style="info",
    icon="sliders",
    layout=widgets.Layout(width="220px", height="38px")
)
btn_run  = widgets.Button(
    description="Update Chart & 2D Drawing",
    button_style="primary",
    icon="refresh",
    layout=widgets.Layout(width="240px", height="38px")
)

out_box = widgets.Output(layout=widgets.Layout(width="100%"))

def sync_section_fields(*args):
    s = w_sec.value
    w_wall.layout.display = "flex" if s == "Hollow Tube" else "none"
    w_d2.layout.display = "flex" if s == "Rectangular Strip" else "none"
    if s == "Hollow Tube":
        w_d1.description = "Outer Ø [mm]:"
    elif s == "Rectangular Strip":
        w_d1.description = "Strip Width b [mm]:"
    else:
        w_d1.description = "Square Side a [mm]:"

w_sec.observe(sync_section_fields, names="value")
sync_section_fields()

def fill_realistic(*args):
    info = get_geometry_and_load(
        w_mat.value, w_sec.value, w_L.value, w_d1.value, w_d2.value, w_wall.value,
        END_CONDITIONS[w_bc.value]
    )
    w_pexp.value = info["P_realistic_exp"]
    render_view()

def render_view(*args):
    with out_box:
        clear_output(wait=True)
        draw_comparison_and_2d_column(
            w_mat.value, w_sec.value, w_bc.value,
            w_L.value, w_d1.value, w_d2.value, w_wall.value, w_pexp.value
        )

btn_real.on_click(fill_realistic)
btn_run.on_click(render_view)

for ctrl in [w_mat, w_sec, w_bc, w_L, w_d1, w_d2, w_wall, w_pexp]:
    ctrl.observe(render_view, names="value")

header_card = widgets.HTML("""
    <div style="
        width: 100%;
        box-sizing: border-box;
        background: linear-gradient(135deg, #0f172a 0%, #1e293b 100%);
        border: 1px solid #334155;
        border-left: 4px solid #38bdf8;
        border-radius: 8px;
        padding: 13px 18px;
        margin-bottom: 10px;
        font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;
    ">
        <div style="font-size:18px;font-weight:700;color:#f8fafc;">Column Buckling — Model vs. Test &amp; Full 2D Drawing</div>
        <div style="font-size:12.5px;color:#94a3b8;margin-top:3px;">
            Compare Euler buckling load against experimental load with full 2D column &amp; cross-section geometry
        </div>
    </div>
""")

controls_card = widgets.VBox([
    widgets.HBox([
        widgets.VBox([w_mat, w_sec, w_bc, w_pexp], layout=widgets.Layout(margin="0 32px 0 0", gap="8px")),
        widgets.VBox([w_L, w_d1, w_d2, w_wall], layout=widgets.Layout(margin="0 32px 0 0", gap="8px")),
        widgets.VBox([btn_real, btn_run], layout=widgets.Layout(justify_content="center", gap="12px", padding="14px 0 0 0"))
    ], layout=widgets.Layout(align_items="flex-start", flex_flow="row wrap", width="100%"))
], layout=widgets.Layout(
    width="100%",
    padding="14px 18px",
    border="1px solid #334155",
    border_radius="8px",
    background_color="#1e293b",
    margin="0 0 10px 0"
))

display(header_card, controls_card, out_box)
fill_realistic()

HTML(value='\n    <div style="\n        width: 100%;\n        box-sizing: border-box;\n        background: lin…

Output(layout=Layout(width='100%'))

In [6]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

# ==========================================
# 1. REALISTIC MATERIAL & BOUNDARY DATA
# ==========================================
MATERIALS = {
    "Paper": {
        "E": 2.2e9, "sigma_c": 22e6, "knockdown": 0.72,
        "desc": "Cardstock/rolled paper (effective E ~ 2.2 GPa). Seams, creases, moisture, and load offset typically reduce test load to ~65-80% of Euler load."
    },
    "Balsa Wood": {
        "E": 3.5e9, "sigma_c": 12e6, "knockdown": 0.78,
        "desc": "Lightweight balsa wood (E ~ 3.5 GPa). Grain direction, density variation, and minor off-center load typically give ~70-85% of Euler load."
    },
    "3D-Printed PLA": {
        "E": 2.8e9, "sigma_c": 45e6, "knockdown": 0.82,
        "desc": "3D-printed PLA column (E ~ 2.8 GPa). Layer bonding, infill pattern, and slight imperfections typically give ~75-90% of Euler load."
    },
}

END_CONDITIONS = {
    "Pinned-Pinned (K=1.0)": 1.0,
    "Fixed-Pinned (K=0.7)":  0.7,
    "Fixed-Fixed (K=0.5)":   0.5,
    "Cantilever (K=2.0)":    2.0,
}

# ==========================================
# 2. GEOMETRY & EULER BUCKLING SOLVER
# ==========================================
def get_geometry_and_load(mat_name, sec_type, L_mm, d1_mm, d2_mm, wall_mm, K_val):
    mat = MATERIALS[mat_name]
    L = L_mm * 1e-3
    Le = K_val * L

    if sec_type == "Hollow Tube":
        D_out = d1_mm * 1e-3
        t = min(wall_mm * 1e-3, 0.45 * D_out)
        D_in = max(D_out - 2.0 * t, 0.0)
        A = (np.pi / 4.0) * (D_out**2 - D_in**2)
        I = (np.pi / 64.0) * (D_out**4 - D_in**4)
        geom_info = {"type": "tube", "D_out_mm": d1_mm, "D_in_mm": D_in * 1e3, "t_mm": t * 1e3}
        geom_text = f"Hollow Tube (Ø {d1_mm:.1f} mm, wall {t*1e3:.2f} mm)"
    elif sec_type == "Rectangular Strip":
        b = d1_mm * 1e-3
        h = d2_mm * 1e-3
        A = b * h
        I = (b * h**3) / 12.0
        geom_info = {"type": "strip", "b_mm": d1_mm, "h_mm": d2_mm}
        geom_text = f"Rectangular Strip ({d1_mm:.1f} x {d2_mm:.1f} mm)"
    else:  # "Solid Square"
        a = d1_mm * 1e-3
        A = a**2
        I = (a**4) / 12.0
        geom_info = {"type": "square", "a_mm": d1_mm}
        geom_text = f"Solid Square ({d1_mm:.1f} x {d1_mm:.1f} mm)"

    k_gyr = np.sqrt(I / max(A, 1e-15))
    slenderness = Le / max(k_gyr, 1e-9)

    P_euler = (np.pi**2 * mat["E"] * I) / max(Le**2, 1e-15)
    P_crush = mat["sigma_c"] * A
    P_model = min(P_euler, P_crush)
    mode = "Euler Buckling" if P_euler <= P_crush else "Compression Yield"

    sec_factor = 0.95 if sec_type == "Hollow Tube" else (0.90 if sec_type == "Rectangular Strip" else 0.96)
    P_realistic_exp = float(np.clip(round(P_model * mat["knockdown"] * sec_factor, 1), 0.5, 300.0))

    return {
        "A_mm2": A * 1e6,
        "I_mm4": I * 1e12,
        "k_mm": k_gyr * 1e3,
        "slenderness": slenderness,
        "P_euler": P_euler,
        "P_crush": P_crush,
        "P_model": P_model,
        "P_realistic_exp": P_realistic_exp,
        "mode": mode,
        "geom_info": geom_info,
        "geom_text": geom_text,
    }

# ==========================================
# 3. SPACED COMPARISON GRAPHIC & 2D DRAWING
# ==========================================
def draw_comparison_and_2d_column(mat_name, sec_type, bc_name, L_mm, d1_mm, d2_mm, wall_mm, P_exp_N):
    K_val = END_CONDITIONS[bc_name]
    info = get_geometry_and_load(mat_name, sec_type, L_mm, d1_mm, d2_mm, wall_mm, K_val)
    P_model = info["P_model"]
    diff_pct = ((P_exp_N - P_model) / max(P_model, 1e-12)) * 100.0

    plt.style.use("seaborn-v0_8-whitegrid")

    # --- SECTION 1: Load Comparison Bar Chart ---
    fig1, ax_bar = plt.subplots(figsize=(14.5, 3.8))
    categories = ["Simple Model (P_model)", "Experimental Load (P_exp)"]
    values = [P_model, P_exp_N]
    colors = ["#3b82f6", "#f97316"]

    bars = ax_bar.bar(categories, values, color=colors, edgecolor="#0f172a", linewidth=1.2, width=0.34, alpha=0.94)
    ax_bar.set_title(f"Buckling Load Comparison — {mat_name} | {info['geom_text']}", fontsize=11.5, fontweight="bold", pad=12)
    ax_bar.set_ylabel("Axial Load [N]", fontsize=10)
    top_y = max(values) * 1.25 if max(values) > 0 else 1.0
    ax_bar.set_ylim(0, top_y)

    for bar, val in zip(bars, values):
        ax_bar.text(
            bar.get_x() + bar.get_width() / 2.0,
            val + top_y * 0.025,
            f"{val:.2f} N",
            ha="center", va="bottom", fontsize=10, fontweight="bold", color="#0f172a"
        )

    ax_bar.text(
        0.99, 0.90,
        f"Difference: {diff_pct:+.1f}%\nGoverning Model: {info['mode']}\nSlenderness (K·L/k): {info['slenderness']:.1f}",
        transform=ax_bar.transAxes, ha="right", va="top", fontsize=9,
        bbox=dict(boxstyle="round,pad=0.45", facecolor="#f8fafc", edgecolor="#cbd5e1")
    )
    plt.tight_layout()
    plt.show()

    # Spacer between Section 1 and Section 2
    display(HTML("<div style='height: 28px;'></div>"))

    # --- SECTION 2: Full 2D Drawing of Selected Column ---
    fig2, (ax_elev, ax_sec) = plt.subplots(1, 2, figsize=(14.5, 4.8), gridspec_kw={"width_ratios": [1.6, 1.0]})
    fig2.subplots_adjust(wspace=0.28)

    s = np.linspace(0.0, 1.0, 200)
    z_mm = s * L_mm

    if "Pinned-Pinned" in bc_name:
        w_norm = np.sin(np.pi * s)
    elif "Fixed-Fixed" in bc_name:
        w_norm = 0.5 * (1.0 - np.cos(2.0 * np.pi * s))
    elif "Cantilever" in bc_name:
        w_norm = 1.0 - np.cos(0.5 * np.pi * s)
    else:
        w_norm = np.sin(np.pi * s) - 0.25 * np.sin(2.0 * np.pi * s)
        w_norm = w_norm / max(np.max(np.abs(w_norm)), 1e-9)

    defl_amp = max(0.08 * L_mm, 5.0)
    half_w = max(d1_mm * 0.5, 2.0)

    ax_elev.plot([-half_w, -half_w], [0, L_mm], color="#64748b", lw=1.5, ls="--", label="Undeformed Column")
    ax_elev.plot([half_w, half_w], [0, L_mm], color="#64748b", lw=1.5, ls="--")
    ax_elev.plot([0, 0], [0, L_mm], color="#94a3b8", lw=1.0, ls=":")

    w_curve = w_norm * defl_amp
    ax_elev.plot(w_curve, z_mm, color="#dc2626", lw=2.6, label="Buckled Centerline")
    ax_elev.plot(w_curve - half_w, z_mm, color="#f97316", lw=1.8, label="Buckled 2D Profile")
    ax_elev.plot(w_curve + half_w, z_mm, color="#f97316", lw=1.8)

    top_x = w_curve[-1]
    ax_elev.annotate("", xy=(top_x, L_mm), xytext=(top_x, L_mm + 0.14 * L_mm),
                     arrowprops=dict(arrowstyle="->", color="#0f172a", lw=2.2))
    ax_elev.text(top_x, L_mm + 0.16 * L_mm, f"P_exp = {P_exp_N:.1f} N", ha="center", va="bottom", fontsize=9.5, fontweight="bold")
    ax_elev.plot([-1.6 * half_w, 1.6 * half_w], [0, 0], color="#0f172a", lw=3.2)
    ax_elev.text(0, -0.08 * L_mm, f"Base: {bc_name}", ha="center", va="top", fontsize=9, fontweight="bold")

    ax_elev.set_title(f"Full 2D Column View (Length L = {L_mm:.0f} mm)", fontsize=11, fontweight="bold", pad=12)
    ax_elev.set_xlabel("Transverse Position / Deflection [mm]", fontsize=9.5)
    ax_elev.set_ylabel("Column Height z [mm]", fontsize=9.5)
    x_lim_elev = max(defl_amp * 1.45, half_w * 2.2)
    ax_elev.set_xlim(-x_lim_elev, x_lim_elev)
    ax_elev.set_ylim(-0.14 * L_mm, L_mm * 1.25)
    ax_elev.legend(loc="upper left", fontsize=8.5, frameon=True, facecolor="white", edgecolor="#cbd5e1")

    g = info["geom_info"]
    if g["type"] == "tube":
        r_out = g["D_out_mm"] / 2.0
        r_in = g["D_in_mm"] / 2.0
        c_out = patches.Circle((0, 0), r_out, facecolor="#3b82f6", edgecolor="#0f172a", lw=1.6, alpha=0.92)
        c_in = patches.Circle((0, 0), r_in, facecolor="white", edgecolor="#0f172a", lw=1.3)
        ax_sec.add_patch(c_out)
        ax_sec.add_patch(c_in)
        lim_s = r_out * 1.35
        sec_title = f"2D Cross-Section: Hollow Tube\nØ_out={g['D_out_mm']:.1f} mm, t={g['t_mm']:.2f} mm"
    elif g["type"] == "strip":
        bw, hh = g["b_mm"], g["h_mm"]
        rect = patches.Rectangle((-bw/2, -hh/2), bw, hh, facecolor="#3b82f6", edgecolor="#0f172a", lw=1.6, alpha=0.92)
        ax_sec.add_patch(rect)
        lim_s = max(bw, hh) * 0.85
        sec_title = f"2D Cross-Section: Rectangular Strip\nb={bw:.1f} mm, h={hh:.1f} mm"
    else:
        aa = g["a_mm"]
        sq = patches.Rectangle((-aa/2, -aa/2), aa, aa, facecolor="#3b82f6", edgecolor="#0f172a", lw=1.6, alpha=0.92)
        ax_sec.add_patch(sq)
        lim_s = aa * 0.85
        sec_title = f"2D Cross-Section: Solid Square\na={aa:.1f} mm"

    ax_sec.set_xlim(-lim_s, lim_s)
    ax_sec.set_ylim(-lim_s, lim_s)
    ax_sec.set_aspect("equal")
    ax_sec.set_title(sec_title, fontsize=10.5, fontweight="bold", pad=12)
    ax_sec.set_xlabel("x [mm]", fontsize=9.5)
    ax_sec.set_ylabel("y [mm]", fontsize=9.5)

    plt.show()

    # Spacer between Section 2 and Section 3
    display(HTML("<div style='height: 28px;'></div>"))

    # --- SECTION 3: Summary Card ---
    if abs(diff_pct) <= 15.0:
        status_html = "<span style='color:#34d399;font-weight:700;'>Close match with simple model</span>"
        border_col = "#10b981"
    elif diff_pct < -15.0:
        status_html = "<span style='color:#f87171;font-weight:700;'>Experimental load is lower than simple model</span>"
        border_col = "#ef4444"
    else:
        status_html = "<span style='color:#fbbf24;font-weight:700;'>Experimental load is higher than simple model</span>"
        border_col = "#f59e0b"

    display(HTML(f"""
    <div style="
        width: 100%;
        box-sizing: border-box;
        background: linear-gradient(135deg, #0f172a 0%, #1e293b 100%);
        border: 1px solid #334155;
        border-left: 4px solid {border_col};
        border-radius: 8px;
        padding: 15px 18px;
        color: #f8fafc;
        font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;
    ">
        <div style="font-size:13.5px;margin-bottom:6px;">
            <b>Result:</b> {status_html} &nbsp;|&nbsp;
            Simple Model: <b>{P_model:.2f} N</b> &nbsp;|&nbsp;
            Experimental: <b>{P_exp_N:.2f} N</b> &nbsp;|&nbsp;
            Diff: <b>{diff_pct:+.1f}%</b>
        </div>
        <div style="font-size:12.5px;color:#cbd5e1;line-height:1.55;">
            <b>Why they differ:</b> {MATERIALS[mat_name]['desc']}
            Ideal Euler theory assumes a perfectly straight column, true centered axial load, and ideal end supports ({bc_name}),
            whereas real tests include slight initial crookedness, load eccentricity, and fixture compliance.
        </div>
    </div>
    """))

# ==========================================
# 4. FULL-WIDTH AESTHETIC ENGLISH UI
# ==========================================
lbl_style = {"description_width": "150px"}
ctrl_layout = widgets.Layout(width="380px")

w_mat  = widgets.Dropdown(options=list(MATERIALS.keys()), value="3D-Printed PLA", description="Material:", style=lbl_style, layout=ctrl_layout)
w_sec  = widgets.Dropdown(options=["Hollow Tube", "Rectangular Strip", "Solid Square"], value="Hollow Tube", description="Section:", style=lbl_style, layout=ctrl_layout)
w_bc   = widgets.Dropdown(options=list(END_CONDITIONS.keys()), value="Pinned-Pinned (K=1.0)", description="End Support:", style=lbl_style, layout=ctrl_layout)
w_pexp = widgets.FloatSlider(value=46.5, min=0.5, max=300.0, step=0.5, description="Exp Load [N]:", readout_format=".1f", style=lbl_style, layout=ctrl_layout)

w_L    = widgets.FloatSlider(value=420.0, min=80.0, max=600.0, step=10.0, description="Length L [mm]:", readout_format=".0f", style=lbl_style, layout=ctrl_layout)
w_d1   = widgets.FloatSlider(value=10.0,  min=4.0,  max=80.0,  step=0.5,  description="Outer Ø [mm]:", readout_format=".1f", style=lbl_style, layout=ctrl_layout)
w_d2   = widgets.FloatSlider(value=2.0,   min=0.5,  max=25.0,  step=0.5,  description="Strip h [mm]:", readout_format=".1f", style=lbl_style, layout=ctrl_layout)
w_wall = widgets.FloatSlider(value=0.9,   min=0.1,  max=5.0,   step=0.1,  description="Tube Wall [mm]:", readout_format=".1f", style=lbl_style, layout=ctrl_layout)

btn_real = widgets.Button(
    description="Fill Realistic P_exp",
    button_style="info",
    icon="sliders",
    layout=widgets.Layout(width="220px", height="38px")
)
btn_run  = widgets.Button(
    description="Update Chart & 2D Drawing",
    button_style="primary",
    icon="refresh",
    layout=widgets.Layout(width="240px", height="38px")
)

out_box = widgets.Output(layout=widgets.Layout(width="100%"))

def sync_section_fields(*args):
    s = w_sec.value
    w_wall.layout.display = "flex" if s == "Hollow Tube" else "none"
    w_d2.layout.display = "flex" if s == "Rectangular Strip" else "none"
    if s == "Hollow Tube":
        w_d1.description = "Outer Ø [mm]:"
    elif s == "Rectangular Strip":
        w_d1.description = "Strip Width b [mm]:"
    else:
        w_d1.description = "Square Side a [mm]:"

w_sec.observe(sync_section_fields, names="value")
sync_section_fields()

def fill_realistic(*args):
    info = get_geometry_and_load(
        w_mat.value, w_sec.value, w_L.value, w_d1.value, w_d2.value, w_wall.value,
        END_CONDITIONS[w_bc.value]
    )
    w_pexp.value = info["P_realistic_exp"]
    render_view()

def render_view(*args):
    with out_box:
        clear_output(wait=True)
        draw_comparison_and_2d_column(
            w_mat.value, w_sec.value, w_bc.value,
            w_L.value, w_d1.value, w_d2.value, w_wall.value, w_pexp.value
        )

btn_real.on_click(fill_realistic)
btn_run.on_click(render_view)

for ctrl in [w_mat, w_sec, w_bc, w_L, w_d1, w_d2, w_wall, w_pexp]:
    ctrl.observe(render_view, names="value")

header_card = widgets.HTML("""
    <div style="
        width: 100%;
        box-sizing: border-box;
        background: linear-gradient(135deg, #0f172a 0%, #1e293b 100%);
        border: 1px solid #334155;
        border-left: 4px solid #38bdf8;
        border-radius: 8px;
        padding: 13px 18px;
        margin-bottom: 14px;
        font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;
    ">
        <div style="font-size:18px;font-weight:700;color:#f8fafc;">Column Buckling — Model vs. Test &amp; Full 2D Drawing</div>
        <div style="font-size:12.5px;color:#94a3b8;margin-top:3px;">
            Compare Euler buckling load against experimental load with full 2D column &amp; cross-section geometry
        </div>
    </div>
""")

controls_card = widgets.VBox([
    widgets.HBox([
        widgets.VBox([w_mat, w_sec, w_bc, w_pexp], layout=widgets.Layout(margin="0 36px 0 0", gap="10px")),
        widgets.VBox([w_L, w_d1, w_d2, w_wall], layout=widgets.Layout(margin="0 36px 0 0", gap="10px")),
        widgets.VBox([btn_real, btn_run], layout=widgets.Layout(justify_content="center", gap="14px", padding="14px 0 0 0"))
    ], layout=widgets.Layout(align_items="flex-start", flex_flow="row wrap", width="100%"))
], layout=widgets.Layout(
    width="100%",
    padding="16px 18px",
    border="1px solid #334155",
    border_radius="8px",
    background_color="#1e293b",
    margin="0 0 22px 0"
))

display(header_card, controls_card, out_box)
fill_realistic()

HTML(value='\n    <div style="\n        width: 100%;\n        box-sizing: border-box;\n        background: lin…

Output(layout=Layout(width='100%'))